In [1]:
# Q0 — installs: ORT pinned to NB1/NB2's version, onnx for graph inspection, pycocotools for mAP
!pip uninstall -y onnxruntime-gpu -q
!pip install -q onnxruntime==1.27.0 onnx pycocotools
import onnxruntime as ort, onnx
print("onnxruntime", ort.__version__, "| onnx", onnx.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 82.9 MB/s eta 0:00:00
onnxruntime 1.27.0 | onnx 1.22.0


In [2]:
# Q1 — results.json helper + environment record (CPU + INT8-relevant ISA flags)
import json, os, sys, glob, subprocess, re, time, io, contextlib
from importlib.metadata import version as pkg_version
import numpy as np, cv2

WORK = "/kaggle/working"
RESULTS = f"{WORK}/results.json"

def record(path, value, cell):
    R = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    d = R
    *parents, leaf = path.split(".")
    for p in parents:
        d = d.setdefault(p, {})
    d[leaf] = {"value": value, "cell": cell}
    json.dump(R, open(RESULTS, "w"), indent=2)
    print(f"[results] {path} = {value}  ({cell})")

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

flags = set(re.search(r"^flags\s*:(.*)$", open("/proc/cpuinfo").read(), re.M).group(1).split())
topo = {k.strip(): v.strip() for k, v in (l.split(":", 1) for l in sh("lscpu").splitlines() if ":" in l)
        if k.strip() in ("Thread(s) per core", "Core(s) per socket", "Socket(s)", "CPU(s)")}
record("env.cpu_model", sh("lscpu | grep 'Model name' | cut -d: -f2 | xargs"), "Q1")
record("env.topology", topo, "Q1")
record("env.isa", {f: (f in flags) for f in ["avx2", "avx512f", "avx512_vnni", "avx_vnni", "amx_int8", "amx_tile"]}, "Q1")
record("env.versions", {"python": sys.version.split()[0], "onnxruntime": ort.__version__, "onnx": onnx.__version__,
                        "opencv": cv2.__version__, "numpy": np.__version__,
                        "pycocotools": pkg_version("pycocotools")}, "Q1")

[results] env.cpu_model = Intel(R) Xeon(R) CPU @ 2.00GHz  (Q1)
[results] env.topology = {'CPU(s)': '4', 'Thread(s) per core': '2', 'Core(s) per socket': '2', 'Socket(s)': '1'}  (Q1)
[results] env.isa = {'avx2': True, 'avx512f': True, 'avx512_vnni': False, 'avx_vnni': False, 'amx_int8': False, 'amx_tile': False}  (Q1)
[results] env.versions = {'python': '3.13.15', 'onnxruntime': '1.27.0', 'onnx': '1.22.0', 'opencv': '4.14.0', 'numpy': '2.1.3', 'pycocotools': '2.0.11'}  (Q1)


In [3]:
# Q2 — locate NB1 models, NB2 class map, COCO val2017 + annotations
def find(name, root="/kaggle/input", skip=("train2017", "test2017", "unlabeled2017", "val2017")):
    hits = []
    for r, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in skip]
        if name in files:
            hits.append(os.path.join(r, name))
    return hits

MODELS = {}
for v in "nsm":
    hits = find(f"yolov8{v}.onnx")
    assert hits, f"yolov8{v}.onnx not found — attach NB1's output"
    MODELS[v] = hits[0]
COCO_ANN = find("instances_val2017.json")[0]
COCO_IMG_DIR = os.path.join(os.path.dirname(os.path.dirname(COCO_ANN)), "val2017")
assert os.path.isdir(COCO_IMG_DIR), COCO_IMG_DIR
map_hits = find("coco80_to_coco91.json")
assert map_hits, "coco80_to_coco91.json not found — attach NB2's output"
COCO91 = json.load(open(map_hits[0]))

print("models :", MODELS)
print("ann    :", COCO_ANN)
print("images :", COCO_IMG_DIR, len(os.listdir(COCO_IMG_DIR)))
print("map    :", map_hits[0], "| person ->", COCO91[0], "| bus ->", COCO91[5])
assert len(COCO91) == 80 and COCO91[0] == 1 and COCO91[5] == 6
record("inputs.models_mb", {v: round(os.path.getsize(p) / 2**20, 1) for v, p in MODELS.items()}, "Q2")

models : {'n': '/kaggle/input/notebooks/rsingh16281/onnx-inference/yolov8n.onnx', 's': '/kaggle/input/notebooks/rsingh16281/onnx-inference/yolov8s.onnx', 'm': '/kaggle/input/notebooks/rsingh16281/onnx-inference/yolov8m.onnx'}
ann    : /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_val2017.json
images : /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017 5000
map    : /kaggle/input/notebooks/rsingh16281/yolov8-onnx-inference-cpp/coco80_to_coco91.json | person -> 1 | bus -> 6
[results] inputs.models_mb = {'n': 12.3, 's': 42.8, 'm': 99.0}  (Q2)


In [4]:
# Q3 — fixed-seed eval (500) and calibration (300) subsets of val2017; overlap must be 0
from pycocotools.coco import COCO
with contextlib.redirect_stdout(io.StringIO()):
    coco_gt = COCO(COCO_ANN)

SEED = 2026
all_ids = sorted(coco_gt.getImgIds())
perm = np.random.default_rng(SEED).permutation(all_ids)
eval_ids  = sorted(int(i) for i in perm[:500])
calib_ids = sorted(int(i) for i in perm[500:800])
overlap = set(eval_ids) & set(calib_ids)
assert len(overlap) == 0, overlap

json.dump({"seed": SEED, "source": "val2017", "n": len(eval_ids), "ids": eval_ids},
          open(f"{WORK}/eval_ids.json", "w"), indent=2)
json.dump({"seed": SEED, "source": "val2017", "n": len(calib_ids), "ids": calib_ids},
          open(f"{WORK}/calib_ids.json", "w"), indent=2)

n_gt = len(coco_gt.getAnnIds(imgIds=eval_ids, iscrowd=False))
n_empty = sum(1 for i in eval_ids if not coco_gt.getAnnIds(imgIds=[i]))
record("subsets", {"seed": SEED, "eval_images": len(eval_ids), "calib_images": len(calib_ids),
                   "overlap": len(overlap), "eval_gt_boxes_non_crowd": n_gt,
                   "eval_images_without_annotations": n_empty}, "Q3")

[results] subsets = {'seed': 2026, 'eval_images': 500, 'calib_images': 300, 'overlap': 0, 'eval_gt_boxes_non_crowd': 3456, 'eval_images_without_annotations': 4}  (Q3)


In [5]:
# Q4 — COCO eval harness (conf=0.001, IoU=0.7, max_det=300) + FP32 yolov8n sanity gate
from pycocotools.cocoeval import COCOeval

def make_sess(path, threads=os.cpu_count()):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])

def letterbox(img, size=640, color=114):
    h, w = img.shape[:2]
    r = size / max(h, w)
    nw, nh = int(w * r), int(h * r)
    canvas = np.full((size, size, 3), color, np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, r, (left, top), ((size - nw) / 2, (size - nh) / 2)   # int paste offset, NB1 float offset

def to_tensor(lb):
    x = cv2.cvtColor(lb, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])

def decode_nms(raw, conf, iou, max_det):
    preds = raw[0].T
    confs, cls = preds[:, 4:].max(axis=1), preds[:, 4:].argmax(axis=1)
    m = confs > conf
    b, c, k = preds[m, :4], confs[m], cls[m]
    xyxy = np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], axis=1)
    keep = []
    for cid in np.unique(k):            # per-class NMS; == NB2 scratch NMS at eval settings (NB2 C25)
        ci = np.where(k == cid)[0]
        xywh = np.stack([xyxy[ci, 0], xyxy[ci, 1], xyxy[ci, 2] - xyxy[ci, 0], xyxy[ci, 3] - xyxy[ci, 1]], axis=1)
        idx = cv2.dnn.NMSBoxes(xywh.tolist(), c[ci].tolist(), score_threshold=0.0, nms_threshold=iou)
        keep.extend(ci[np.asarray(idx, dtype=int).flatten()].tolist())
    keep = np.asarray(keep, dtype=int)
    if max_det and len(keep) > max_det:
        keep = keep[np.argsort(-c[keep], kind="stable")[:max_det]]
    return xyxy[keep], c[keep], k[keep]

def scale_to_orig(xyxy, r, off, w, h):
    b = xyxy.astype(np.float32).copy()
    b[:, [0, 2]] = np.clip((b[:, [0, 2]] - off[0]) / r, 0, w)
    b[:, [1, 3]] = np.clip((b[:, [1, 3]] - off[1]) / r, 0, h)
    return b

def run_eval(model_path, ids, conf=0.001, iou=0.7, max_det=300, threads=os.cpu_count()):
    """Returns COCO-format dets for both scale-back offsets from ONE inference pass."""
    sess = make_sess(model_path, threads)
    i_name, o_name = sess.get_inputs()[0].name, sess.get_outputs()[0].name
    dets = {"int_pad": [], "float_pad": []}
    t0 = time.time()
    for iid in ids:
        info = coco_gt.loadImgs(iid)[0]
        img = cv2.imread(f"{COCO_IMG_DIR}/{info['file_name']}")
        h, w = img.shape[:2]
        lb, r, ipad, fpad = letterbox(img)
        raw = sess.run([o_name], {i_name: to_tensor(lb)})[0]
        xyxy, c, k = decode_nms(raw, conf, iou, max_det)
        for key, off in (("int_pad", ipad), ("float_pad", fpad)):
            bx = scale_to_orig(xyxy, r, off, w, h)
            dets[key] += [{"image_id": int(iid), "category_id": COCO91[int(k[j])],
                           "bbox": [float(bx[j, 0]), float(bx[j, 1]),
                                    float(bx[j, 2] - bx[j, 0]), float(bx[j, 3] - bx[j, 1])],
                           "score": float(c[j])} for j in range(len(c))]
    print(f"{os.path.basename(model_path)}: {len(ids)} images in {time.time() - t0:.0f}s, "
          f"{len(dets['int_pad'])} dets")
    return dets

def coco_map(dets, ids):
    if not dets:                                    # 0 detections is a valid result: mAP = 0 (loadRes can't take [])
        return {"map50_95": 0.0, "map50": 0.0, "stats": [0.0] * 12, "n_dets": 0}
    dets = [dict(d) for d in dets]                  # loadRes mutates dicts; keep caller's copy clean
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(coco_gt, coco_gt.loadRes(dets), "bbox")
        E.params.imgIds = ids
        E.evaluate(); E.accumulate(); E.summarize()
    return {"map50_95": float(E.stats[0]), "map50": float(E.stats[1]),
            "stats": [float(s) for s in E.stats], "n_dets": len(dets)}
EVAL_CFG = {"conf": 0.001, "iou": 0.7, "max_det": 300}
record("eval.settings", dict(EVAL_CFG, letterbox="square 640, pad 114", nms="per-class greedy"), "Q4")

fp32_n = run_eval(MODELS["n"], eval_ids, **EVAL_CFG)
json.dump(fp32_n["int_pad"], open(f"{WORK}/dets_fp32_n.json", "w"))
m_fix = coco_map(fp32_n["int_pad"],   eval_ids)
m_nb1 = coco_map(fp32_n["float_pad"], eval_ids)
print(f"yolov8n FP32  mAP50-95={m_fix['map50_95']:.4f}  mAP50={m_fix['map50']:.4f}   (integer paste offset)")
print(f"              mAP50-95={m_nb1['map50_95']:.4f}  mAP50={m_nb1['map50']:.4f}   (NB1 float offset)")

record("map.fp32.n", m_fix, "Q4")
record("map.halfpixel_ablation_n", {"int_pad_map50_95": m_fix["map50_95"], "nb1_float_pad_map50_95": m_nb1["map50_95"],
                                    "delta": m_fix["map50_95"] - m_nb1["map50_95"]}, "Q4")
assert m_fix["map50_95"] >= 0.30, "SANITY FAIL: mAP < 0.30 — check class mapping / box format before going further"
print("GATE 4a PASS (FP32 yolov8n sanity)")

[results] eval.settings = {'conf': 0.001, 'iou': 0.7, 'max_det': 300, 'letterbox': 'square 640, pad 114', 'nms': 'per-class greedy'}  (Q4)
yolov8n.onnx: 500 images in 48s, 69761 dets
yolov8n FP32  mAP50-95=0.3812  mAP50=0.5138   (integer paste offset)
              mAP50-95=0.3802  mAP50=0.5145   (NB1 float offset)
[results] map.fp32.n = {'map50_95': 0.38119653652033014, 'map50': 0.5137662020270171, 'stats': [0.38119653652033014, 0.5137662020270171, 0.40987208284781795, 0.19166871856534892, 0.4435667829618866, 0.5506611192931335, 0.31273213590978816, 0.5079539723766369, 0.5613131078585618, 0.35808514146941856, 0.6306393037462437, 0.7150749669802641], 'n_dets': 69761}  (Q4)
[results] map.halfpixel_ablation_n = {'int_pad_map50_95': 0.38119653652033014, 'nb1_float_pad_map50_95': 0.3802413272080251, 'delta': 0.0009552093123050276}  (Q4)
GATE 4a PASS (FP32 yolov8n sanity)


In [6]:
# Q5 — FP32 mAP for s, m and model-only CPU latency (3 warm-up, 50 timed) for n, s, m at 1 and all threads
bench_img = coco_gt.loadImgs(eval_ids[0])[0]["file_name"]
x_bench = to_tensor(letterbox(cv2.imread(f"{COCO_IMG_DIR}/{bench_img}"))[0])

def bench_model_only(path, threads, warm=3, iters=50):
    sess = make_sess(path, threads)
    name = sess.get_inputs()[0].name
    ts = []
    for i in range(warm + iters):
        t0 = time.perf_counter(); sess.run(None, {name: x_bench})
        if i >= warm:
            ts.append((time.perf_counter() - t0) * 1e3)
    return {"mean_ms": float(np.mean(ts)), "std_ms": float(np.std(ts))}

THREADS = [1, os.cpu_count()]
record("latency.protocol", {"warmup": 3, "iters": 50, "input": bench_img, "threads": THREADS,
                            "measures": "session.run only (model-only)"}, "Q5")

fp32 = {"n": m_fix}
for v in "sm":
    d = run_eval(MODELS[v], eval_ids, **EVAL_CFG)
    json.dump(d["int_pad"], open(f"{WORK}/dets_fp32_{v}.json", "w"))
    fp32[v] = coco_map(d["int_pad"], eval_ids)
    record(f"map.fp32.{v}", fp32[v], "Q5")

lat_fp32 = {v: {f"t{t}": bench_model_only(MODELS[v], t) for t in THREADS} for v in "nsm"}
record("latency.fp32", lat_fp32, "Q5")

sizes = {v: os.path.getsize(MODELS[v]) / 2**20 for v in "nsm"}
print(f"\n{'model':8s} {'MB':>6s} {'mAP50-95':>9s} {'mAP50':>7s} " +
      " ".join(f"{'lat t'+str(t)+' ms':>16s}" for t in THREADS))
for v in "nsm":
    print(f"yolov8{v:2s} {sizes[v]:6.1f} {fp32[v]['map50_95']:9.4f} {fp32[v]['map50']:7.4f} " +
          " ".join(f"{lat_fp32[v][f't{t}']['mean_ms']:9.1f}±{lat_fp32[v][f't{t}']['std_ms']:<5.1f}" for t in THREADS))
assert fp32["n"]["map50_95"] < fp32["s"]["map50_95"] < fp32["m"]["map50_95"], "n<s<m ordering broken"
print("GATE 4b PASS (FP32 baselines, ordering n < s < m)")

[results] latency.protocol = {'warmup': 3, 'iters': 50, 'input': '000000000885.jpg', 'threads': [1, 4], 'measures': 'session.run only (model-only)'}  (Q5)
yolov8s.onnx: 500 images in 85s, 55709 dets
[results] map.fp32.s = {'map50_95': 0.4629295209677349, 'map50': 0.6022680275858161, 'stats': [0.4629295209677349, 0.6022680275858161, 0.49491769020215126, 0.2877543218860449, 0.5109556060787601, 0.6322224249081432, 0.3628140417665803, 0.5777432699855816, 0.6281398290293522, 0.45306753691015966, 0.6711640416189878, 0.7491101303406215], 'n_dets': 55709}  (Q5)
yolov8m.onnx: 500 images in 210s, 46540 dets
[results] map.fp32.m = {'map50_95': 0.5116671341317693, 'map50': 0.6619467911294508, 'stats': [0.5116671341317693, 0.6619467911294508, 0.5432641359732772, 0.36229351139903626, 0.5564248718690964, 0.6837171285237911, 0.38999802567695563, 0.6242142658262583, 0.6698744908501876, 0.5340739246423276, 0.6937861991704549, 0.8020417973670338], 'n_dets': 46540}  (Q5)
[results] latency.fp32 = {'n': {'t

In [7]:
# Q6 — onnxruntime.quantization pre-process on yolov8n, then prove the graph still computes the same thing
import inspect
from collections import Counter
from onnxruntime.quantization.shape_inference import quant_pre_process
print("quant_pre_process", inspect.signature(quant_pre_process))

N_PREP = f"{WORK}/yolov8n_prep.onnx"
quant_pre_process(MODELS["n"], N_PREP)

ops_before = Counter(n.op_type for n in onnx.load(MODELS["n"]).graph.node)
ops_after  = Counter(n.op_type for n in onnx.load(N_PREP).graph.node)
print("nodes before:", sum(ops_before.values()), dict(ops_before.most_common(8)))
print("nodes after :", sum(ops_after.values()),  dict(ops_after.most_common(8)))

y_orig = make_sess(MODELS["n"]).run(None, {"images": x_bench})[0]
y_prep = make_sess(N_PREP).run(None, {"images": x_bench})[0]
d_box, d_score = np.abs(y_orig[:, :4] - y_prep[:, :4]).max(), np.abs(y_orig[:, 4:] - y_prep[:, 4:]).max()
print(f"prep vs original: max box diff {d_box:.3e}  max score diff {d_score:.3e}")

record("quant.preprocess_n", {"nodes_before": sum(ops_before.values()), "nodes_after": sum(ops_after.values()),
                              "max_box_diff": float(d_box), "max_score_diff": float(d_score),
                              "size_mb": round(os.path.getsize(N_PREP) / 2**20, 2)}, "Q6")
assert d_score < 1e-3 and d_box < 1e-2, "pre-processed graph diverges — stop"

quant_pre_process (input_model: str | pathlib._local.Path | onnx.onnx_ml_pb2.ModelProto | None = None, output_model_path: str | pathlib._local.Path | None = None, skip_optimization: bool = False, skip_onnx_shape: bool = False, skip_symbolic_shape: bool = False, auto_merge: bool = False, int_max: int = 2147483647, guess_output_rank: bool = False, verbose: int = 0, save_as_external_data: bool = False, all_tensors_to_one_file: bool = False, external_data_location: str | None = None, external_data_size_threshold: int = 1024, **deprecated_kwargs) -> None
nodes before: 234 {'Conv': 64, 'Sigmoid': 58, 'Mul': 58, 'Concat': 17, 'Split': 8, 'Add': 8, 'Reshape': 8, 'MaxPool': 3}
nodes after : 234 {'Conv': 64, 'Sigmoid': 58, 'Mul': 58, 'Concat': 17, 'Split': 8, 'Add': 8, 'Reshape': 8, 'MaxPool': 3}
prep vs original: max box diff 0.000e+00  max score diff 0.000e+00
[results] quant.preprocess_n = {'nodes_before': 234, 'nodes_after': 234, 'max_box_diff': 0.0, 'max_score_diff': 0.0, 'size_mb': 12.24} 

In [8]:
# Q6b — per-channel QDQ needs opset >= 13 (Q/DQ 'axis' attribute). Convert NB1's opset-12 export -> 13,
#       re-run pre-process, and prove the graph still computes the same thing.
from onnx import version_converter

opset_of = lambda p: {o.domain or "ai.onnx": o.version for o in onnx.load(p).opset_import}
print("NB1 export opset:", opset_of(MODELS["n"]))

m13 = version_converter.convert_version(onnx.load(MODELS["n"]), 13)
onnx.checker.check_model(m13)
N13 = f"{WORK}/yolov8n_op13.onnx"
onnx.save(m13, N13)

N_PREP = f"{WORK}/yolov8n_op13_prep.onnx"          # every later quantize() call uses this
quant_pre_process(N13, N_PREP)
print("prep opset    :", opset_of(N_PREP))

ops12 = Counter(n.op_type for n in onnx.load(MODELS["n"]).graph.node)
ops13 = Counter(n.op_type for n in onnx.load(N_PREP).graph.node)
y13 = make_sess(N_PREP).run(None, {"images": x_bench})[0]
d_box   = float(np.abs(y_orig[:, :4] - y13[:, :4]).max())
d_score = float(np.abs(y_orig[:, 4:] - y13[:, 4:]).max())
print(f"op-type changes: {dict((ops13 - ops12) + (ops12 - ops13)) or 'none'}")
print(f"opset13 prep vs original: max box diff {d_box:.3e}  max score diff {d_score:.3e}")

record("quant.opset13_conversion_n", {"from": opset_of(MODELS["n"]), "to": opset_of(N_PREP),
       "nodes": sum(ops13.values()), "max_box_diff": d_box, "max_score_diff": d_score}, "Q6b")
assert d_score < 1e-3 and d_box < 1e-2, "opset conversion changed the outputs — stop"

NB1 export opset: {'ai.onnx': 12}
prep opset    : {'ai.onnx': 13, 'ai.onnx.ml': 5, 'ai.onnx.training': 1, 'ai.onnx.preview.training': 1, 'com.microsoft': 1, 'com.microsoft.experimental': 1, 'com.microsoft.nchwc': 1, 'org.pytorch.aten': 1}
op-type changes: {'Transpose': 1}
opset13 prep vs original: max box diff 0.000e+00  max score diff 0.000e+00
[results] quant.opset13_conversion_n = {'from': {'ai.onnx': 12}, 'to': {'ai.onnx': 13, 'ai.onnx.ml': 5, 'ai.onnx.training': 1, 'ai.onnx.preview.training': 1, 'com.microsoft': 1, 'com.microsoft.experimental': 1, 'com.microsoft.nchwc': 1, 'org.pytorch.aten': 1}, 'nodes': 233, 'max_box_diff': 0.0, 'max_score_diff': 0.0}  (Q6b)


In [9]:
# Q7 — calibration reader: same letterbox + to_tensor as the eval pipeline, 300 calib images, rewindable
from onnxruntime.quantization import CalibrationDataReader

class YoloCalibReader(CalibrationDataReader):
    def __init__(self, ids, input_name="images"):
        self.ids, self.input_name, self.i = list(ids), input_name, 0
    def get_next(self):
        if self.i >= len(self.ids):
            return None
        info = coco_gt.loadImgs(self.ids[self.i])[0]
        self.i += 1
        img = cv2.imread(f"{COCO_IMG_DIR}/{info['file_name']}")
        return {self.input_name: to_tensor(letterbox(img)[0])}
    def rewind(self):
        self.i = 0

rd = YoloCalibReader(calib_ids)
t0, n, lo, hi, shapes = time.time(), 0, 1.0, 0.0, set()
while (b := rd.get_next()) is not None:
    x = b["images"]; n += 1
    shapes.add((x.shape, str(x.dtype))); lo, hi = min(lo, float(x.min())), max(hi, float(x.max()))
print(f"{n} batches in {time.time() - t0:.1f}s | shapes/dtypes {shapes} | value range [{lo}, {hi}]")
rd.rewind(); assert rd.get_next() is not None and rd.i == 1

record("quant.calib_reader", {"batches": n, "shape": str(list(shapes)[0][0]), "dtype": list(shapes)[0][1],
                              "min": lo, "max": hi, "same_preprocess_as_eval": True}, "Q7")
assert n == 300 and len(shapes) == 1 and lo >= 0.0 and hi <= 1.0

300 batches in 5.6s | shapes/dtypes {((1, 3, 640, 640), 'float32')} | value range [0.0, 1.0]
[results] quant.calib_reader = {'batches': 300, 'shape': '(1, 3, 640, 640)', 'dtype': 'float32', 'min': 0.0, 'max': 1.0, 'same_preprocess_as_eval': True}  (Q7)


In [10]:
# Q8 — (a) RAM the calibrator would hold for 300 images, (b) what the installed quantization API actually does
import onnxruntime.quantization as oq
import onnxruntime.quantization.quantize as oq_quant
import onnxruntime.quantization.calibrate as oq_cal

# (a) sum of all float32 intermediate tensors per image, from the shape-inferred prep model
m_inf = onnx.shape_inference.infer_shapes(onnx.load(N_PREP))
elems, n_t = 0, 0
for vi in list(m_inf.graph.value_info) + list(m_inf.graph.output):
    tt = vi.type.tensor_type
    dims = [d.dim_value for d in tt.shape.dim]
    if tt.elem_type == onnx.TensorProto.FLOAT and dims and all(d > 0 for d in dims):
        elems += int(np.prod(dims)); n_t += 1
per_img_mb = elems * 4 / 2**20
meminfo = dict(l.split(":", 1) for l in open("/proc/meminfo").read().splitlines())
avail_gb = int(meminfo["MemAvailable"].split()[0]) / 2**20
total_gb = int(meminfo["MemTotal"].split()[0]) / 2**20
need_gb = per_img_mb * len(calib_ids) / 1024
print(f"{n_t} float tensors, {per_img_mb:.0f} MB of activations per image "
      f"-> ~{need_gb:.1f} GB for {len(calib_ids)} calibration images")
print(f"RAM: total {total_gb:.1f} GB, available now {avail_gb:.1f} GB")
record("quant.calib_memory_estimate", {"tensors": n_t, "mb_per_image": round(per_img_mb, 1),
       "gb_for_all_calib": round(need_gb, 1), "ram_total_gb": round(total_gb, 1),
       "ram_available_gb": round(avail_gb, 1)}, "Q8")

# (b) installed API — read it, don't assume it
print("\nquantize_static", inspect.signature(oq.quantize_static))
print("CalibrationMethod:", [c.name for c in oq.CalibrationMethod])
print("create_calibrator is looked up inside quantize module:", hasattr(oq_quant, "create_calibrator"))
print("\nextra_options documented in quantize_static that mention Calib:")
for line in (oq.quantize_static.__doc__ or "").splitlines():
    if "Calib" in line:
        print("   ", line.strip())
for cls in ("MinMaxCalibrater", "HistogramCalibrater"):
    print(f"\n──── {cls}.collect_data (installed source) ────")
    print(inspect.getsource(getattr(oq_cal, cls).collect_data))

244 float tensors, 235 MB of activations per image -> ~69.0 GB for 300 calibration images
RAM: total 31.3 GB, available now 29.4 GB
[results] quant.calib_memory_estimate = {'tensors': 244, 'mb_per_image': 235.5, 'gb_for_all_calib': 69.0, 'ram_total_gb': 31.3, 'ram_available_gb': 29.4}  (Q8)

quantize_static (model_input: 'str | Path | onnx.ModelProto', model_output: 'str | Path', calibration_data_reader: 'CalibrationDataReader | None' = None, quant_format=<QuantFormat.QDQ: 1>, op_types_to_quantize=None, per_channel=False, reduce_range=False, activation_type=<QuantType.QInt8: 0>, weight_type=<QuantType.QInt8: 0>, nodes_to_quantize=None, nodes_to_exclude=None, use_external_data_format=False, calibrate_method=<CalibrationMethod.MinMax: 0>, calibration_providers=None, extra_options=None, calibration_cache_path: 'str | Path | None' = None)
CalibrationMethod: ['MinMax', 'Entropy', 'Percentile', 'Distribution']
create_calibrator is looked up inside quantize module: False

extra_options docume

In [11]:
# Q9 — bounded-RAM chunked calibration; verified vs single-pass on a 40-image calibration subset
from onnxruntime.quantization import quantize_static, QuantFormat, QuantType, CalibrationMethod

# ── verify the two assumptions in the installed source ──
print(inspect.getsource(oq_cal.MinMaxCalibrater.clear_collected_data))
mm_merges = "merge_range" in inspect.getsource(oq_cal.MinMaxCalibrater.compute_data)
print("MinMax.compute_data merges with earlier ranges:", mm_merges)
for name in ("EntropyCalibrater", "PercentileCalibrater"):
    print(f"{name} overrides collect_data:", "collect_data" in getattr(oq_cal, name).__dict__)
assert mm_merges, "MinMax would not merge across chunks — stop"
assert not any("collect_data" in getattr(oq_cal, n).__dict__ for n in ("EntropyCalibrater", "PercentileCalibrater"))

# ── chunking wrapper: feeds the reader to the original collect_data k images at a time ──
CHUNK = {"k": None, "log": []}

class _Chunk(CalibrationDataReader):
    def __init__(self, first, src, k):
        self.first, self.src, self.left = first, src, k - 1
    def get_next(self):
        if self.first is not None:
            b, self.first = self.first, None
            return b
        if self.left <= 0:
            return None
        self.left -= 1
        return self.src.get_next()

def _make_collect(orig):
    def collect_data(self, data_reader):
        if not CHUNK["k"]:
            return orig(self, data_reader)
        n = 0
        while (first := data_reader.get_next()) is not None:   # peek, so we never pass an empty chunk
            orig(self, _Chunk(first, data_reader, CHUNK["k"])); n += 1
        CHUNK["log"].append((type(self).__name__, n))
    return collect_data

if not hasattr(oq_cal, "_ORIG_COLLECT"):                       # patch once, even if this cell is re-run
    oq_cal._ORIG_COLLECT = {c: c.collect_data for c in (oq_cal.MinMaxCalibrater, oq_cal.HistogramCalibrater)}
for cls, orig in oq_cal._ORIG_COLLECT.items():
    cls.collect_data = _make_collect(orig)

# ── quantize helper (config B family: QDQ, per-channel S8 weights, U8 activations) ──
def quantize(method, reader, out_path, nodes_to_exclude=None):
    t0 = time.time()
    quantize_static(N_PREP, out_path, calibration_data_reader=reader, quant_format=QuantFormat.QDQ,
                    per_channel=True, activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                    calibrate_method=method, nodes_to_exclude=nodes_to_exclude or [])
    return time.time() - t0

def act_scales(path):
    m = onnx.load(path)
    init = {i.name: onnx.numpy_helper.to_array(i) for i in m.graph.initializer}
    return {n.output[0]: init[n.input[1]] for n in m.graph.node
            if n.op_type == "QuantizeLinear" and n.input[1] in init}   # Q nodes = activation quantization

def scale_diff(a, b):
    sa, sb = act_scales(a), act_scales(b)
    assert sa.keys() == sb.keys(), "different quantized tensors"
    rel = [float(np.max(np.abs(sa[k] - sb[k]) / np.maximum(np.abs(sb[k]), 1e-12))) for k in sa]
    return {"n_act_scales": len(sa), "max_rel_diff": max(rel), "median_rel_diff": float(np.median(rel))}

TMP = f"{WORK}/chunk_check"; os.makedirs(TMP, exist_ok=True)
sub40 = calib_ids[:40]
chk = {}
for name, meth in [("MinMax", CalibrationMethod.MinMax), ("Entropy", CalibrationMethod.Entropy),
                   ("Percentile", CalibrationMethod.Percentile)]:
    CHUNK["k"] = None
    t_single = quantize(meth, YoloCalibReader(sub40), f"{TMP}/{name}_single.onnx")
    CHUNK["k"], CHUNK["log"] = 10, []
    t_chunk = quantize(meth, YoloCalibReader(sub40), f"{TMP}/{name}_chunk10.onnx")
    d = scale_diff(f"{TMP}/{name}_chunk10.onnx", f"{TMP}/{name}_single.onnx")
    d.update(chunks=CHUNK["log"][-1][1], sec_single=round(t_single), sec_chunked=round(t_chunk))
    chk[name] = d
    print(f"{name:10s} {d}")
CHUNK["k"] = None
record("quant.chunked_calibration_check_40img", chk, "Q9")
assert chk["MinMax"]["max_rel_diff"] == 0.0, "chunked MinMax != single-pass — stop"
print("Chunked MinMax is bit-identical to single-pass; histogram-method drift reported above")

    def clear_collected_data(self):
        self.intermediate_outputs = []

MinMax.compute_data merges with earlier ranges: True
EntropyCalibrater overrides collect_data: False
PercentileCalibrater overrides collect_data: False
MinMax     {'n_act_scales': 242, 'max_rel_diff': 0.0, 'median_rel_diff': 0.0, 'chunks': 4, 'sec_single': 8, 'sec_chunked': 7}
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 242
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 242
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
Entropy    {'n_act_scales': 242, 'max_rel_diff': 0.41559773683547974, 'median_rel_diff': 0.0, 'chunks': 4, 'sec_single': 41, 'sec_chunked': 35}
Finding optimal threshold for each tensor using 'percentile' algorithm

In [12]:
# Q10 — (a) which tensors moved under chunked Entropy; (b) WHY full-INT8 gives 0 detections, from printed numbers
def act_scale_rel(a, b):
    """Per-tensor max relative difference of activation (QuantizeLinear) scales between two QDQ models."""
    sa, sb = act_scales(a), act_scales(b)
    return {k: float(np.max(np.abs(sa[k] - sb[k]) / np.maximum(np.abs(sb[k]), 1e-12))) for k in sa}

rel = act_scale_rel(f"{TMP}/Entropy_chunk10.onnx", f"{TMP}/Entropy_single.onnx")
moved = sorted(((v, k) for k, v in rel.items() if v > 0.01), reverse=True)
print(f"(a) Entropy chunked vs single: {len(moved)} / {len(rel)} tensors moved >1%; top:")
for v, k in moved[:8]:
    print(f"   {v:7.3f}  {k}")
record("quant.entropy_chunk_moved_tensors_40img", {"moved_gt_1pct": len(moved), "total": len(rel),
       "top": [(k, round(v, 4)) for v, k in moved[:8]]}, "Q10")

# (b) find the quantization params that sit on output0
q_mm = f"{TMP}/MinMax_single.onnx"
mq = onnx.load(q_mm)
init = {i.name: onnx.numpy_helper.to_array(i) for i in mq.graph.initializer}
producer = {o: n for n in mq.graph.node for o in n.output}
dq = producer[mq.graph.output[0].name]
print(f"\n(b) output0 is produced by: {dq.op_type} '{dq.name}'")
assert dq.op_type == "DequantizeLinear", "unexpected graph tail — paste this output"
q_node = producer[dq.input[0]]
src = producer[q_node.input[0]]
s_out = float(init[dq.input[1]])
zp_out = int(init[dq.input[2]]) if len(dq.input) > 2 else 0
print(f"    quantized tensor '{q_node.input[0]}' comes from {src.op_type} '{src.name}'")
print(f"    output0 scale = {s_out:.4f}, zero point = {zp_out}  ->  any value within ±{s_out/2:.3f} of 0 becomes 0")

y_fp = make_sess(N_PREP).run(None, {"images": x_bench})[0]
y_q  = make_sess(q_mm).run(None, {"images": x_bench})[0]
n50_fp = len(run_eval(N_PREP, eval_ids[:50], **EVAL_CFG)["int_pad"])
n50_q  = len(run_eval(q_mm,   eval_ids[:50], **EVAL_CFG)["int_pad"])
diag = {
    "fp32_box_range":   [float(y_fp[:, :4].min()), float(y_fp[:, :4].max())],
    "fp32_score_range": [float(y_fp[:, 4:].min()), float(y_fp[:, 4:].max())],
    "output_concat_node": src.name, "output_scale": s_out, "output_zero_point": zp_out,
    "fp32_scores_above_half_step": int((y_fp[:, 4:] > s_out / 2).sum()),
    "int8_score_unique_values": np.unique(y_q[:, 4:])[:10].tolist(),
    "int8_max_score": float(y_q[:, 4:].max()),
    "int8_box_mean_abs_err_px": float(np.abs(y_q[:, :4] - y_fp[:, :4]).mean()),
    "dets_50img_fp32": n50_fp, "dets_50img_int8_full": n50_q,
}
for k, v in diag.items():
    print(f"    {k}: {v}")
record("quant.full_int8_output_diagnosis", diag, "Q10")

(a) Entropy chunked vs single: 83 / 242 tensors moved >1%; top:
     0.416  /model.22/cv3.0/cv3.0.1/act/Mul_output_0_QuantizeLinear_Output
     0.401  /model.21/cv1/act/Mul_output_0_QuantizeLinear_Output
     0.401  /model.21/Split_output_1_QuantizeLinear_Output
     0.401  /model.21/Split_output_0_QuantizeLinear_Output
     0.352  /model.22/cv2.1/cv2.1.0/act/Mul_output_0_QuantizeLinear_Output
     0.319  /model.22/cv2.1/cv2.1.1/act/Mul_output_0_QuantizeLinear_Output
     0.288  /model.21/m.0/cv1/act/Mul_output_0_QuantizeLinear_Output
     0.241  /model.19/act/Mul_output_0_QuantizeLinear_Output
[results] quant.entropy_chunk_moved_tensors_40img = {'moved_gt_1pct': 83, 'total': 242, 'top': [('/model.22/cv3.0/cv3.0.1/act/Mul_output_0_QuantizeLinear_Output', 0.4156), ('/model.21/cv1/act/Mul_output_0_QuantizeLinear_Output', 0.4012), ('/model.21/Split_output_1_QuantizeLinear_Output', 0.4012), ('/model.21/Split_output_0_QuantizeLinear_Output', 0.4012), ('/model.22/cv2.1/cv2.1.0/act/Mul_output

In [13]:
# Q11 — head tail discovery + build configs B, E1, E2 (MinMax, 300 calib images, chunked k=25)
g = onnx.load(N_PREP).graph
names = {n.name for n in g.node}
tail = [n for n in g.node if n.name.startswith("/model.22/") and "/cv2" not in n.name and "/cv3" not in n.name]
print(f"model.22 decode tail ({len(tail)} nodes, everything after the cv2/cv3 head convs):")
for n in tail:
    print(f"   {n.op_type:12s} {n.name}")

EXCL = {"B": [], "E1": ["/model.22/Concat_3"], "E2": [n.name for n in tail]}
assert all(x in names for v in EXCL.values() for x in v), "exclusion names not found in N_PREP"
record("quant.exclusion_sets", {k: v for k, v in EXCL.items()}, "Q11")

QDIR = f"{WORK}/int8"; os.makedirs(QDIR, exist_ok=True)
QPATH = {k: f"{QDIR}/n_{k}_minmax.onnx" for k in EXCL}
CHUNK["k"] = 25
build = {}
for k, excl in EXCL.items():
    CHUNK["log"] = []
    secs = quantize(CalibrationMethod.MinMax, YoloCalibReader(calib_ids), QPATH[k], nodes_to_exclude=excl)
    n_q = sum(1 for n in onnx.load(QPATH[k]).graph.node if n.op_type == "QuantizeLinear")
    build[k] = {"seconds": round(secs), "chunks": CHUNK["log"][-1][1], "quantize_nodes": n_q,
                "excluded": len(excl), "size_mb": round(os.path.getsize(QPATH[k]) / 2**20, 2)}
    print(f"{k:3s} built in {secs:.0f}s, {build[k]}")
CHUNK["k"] = None
record("quant.build_minmax_300", build, "Q11")

model.22 decode tail (24 nodes, everything after the cv2/cv3 head convs):
   Reshape      /model.22/Reshape_2
   Reshape      /model.22/Reshape_1
   Reshape      /model.22/Reshape
   Concat       /model.22/Concat
   Reshape      /model.22/dfl/Reshape
   Softmax      /model.22/dfl/Softmax
   Transpose    /model.22/dfl/Transpose_1
   Conv         /model.22/dfl/conv/Conv
   Reshape      /model.22/dfl/Reshape_1
   Slice        /model.22/Slice_1
   Add          /model.22/Add_1
   Slice        /model.22/Slice
   Sub          /model.22/Sub
   Add          /model.22/Add_2
   Div          /model.22/Div_1
   Sub          /model.22/Sub_1
   Concat       /model.22/Concat_2
   Mul          /model.22/Mul_2
   Reshape      /model.22/Reshape_5
   Reshape      /model.22/Reshape_4
   Reshape      /model.22/Reshape_3
   Concat       /model.22/Concat_1
   Sigmoid      /model.22/Sigmoid
   Concat       /model.22/Concat_3
[results] quant.exclusion_sets = {'B': [], 'E1': ['/model.22/Concat_3'], 'E2': ['/mode

In [14]:
# Q12 — ablation rows: size, CPU latency (t1, t4), mAP50-95, mAP50, output drift vs FP32 on 50 images
x50 = [to_tensor(letterbox(cv2.imread(f"{COCO_IMG_DIR}/{coco_gt.loadImgs(i)[0]['file_name']}"))[0])
       for i in eval_ids[:50]]
s_fp = make_sess(N_PREP)
y50_fp = [s_fp.run(None, {"images": x})[0] for x in x50]

def drift50(path):
    s = make_sess(path); b, c = [], []
    for x, yf in zip(x50, y50_fp):
        y = s.run(None, {"images": x})[0]
        b.append(np.abs(y[:, :4] - yf[:, :4]).mean()); c.append(np.abs(y[:, 4:] - yf[:, 4:]).mean())
    return {"box_mae_px": float(np.mean(b)), "score_mae": float(np.mean(c))}

def evaluate_config(name, path, desc, cell):
    d = run_eval(path, eval_ids, **EVAL_CFG)
    json.dump(d["int_pad"], open(f"{WORK}/dets_{name}.json", "w"))
    mp = coco_map(d["int_pad"], eval_ids)
    row = {"desc": desc, "path": os.path.basename(path), "size_mb": round(os.path.getsize(path) / 2**20, 2),
           "map50_95": mp["map50_95"], "map50": mp["map50"], "n_dets": len(d["int_pad"]),
           "latency": {f"t{t}": bench_model_only(path, t) for t in THREADS}, "drift_50img": drift50(path)}
    record(f"ablation.{name}", row, cell)
    return row

rows = {"A": evaluate_config("A", N_PREP, "FP32 (opset13, pre-processed)", "Q12")}
desc = {"B": "INT8 MinMax, all nodes", "E1": "INT8 MinMax, final Concat in FP32",
        "E2": "INT8 MinMax, whole model.22 decode tail in FP32"}
for k in ["B", "E1", "E2"]:
    rows[k] = evaluate_config(k, QPATH[k], desc[k], "Q12")

print(f"\n{'cfg':4s} {'MB':>6s} {'mAP50-95':>9s} {'mAP50':>7s} {'dets':>7s} {'t1 ms':>13s} {'t4 ms':>13s} {'box MAE':>8s} {'score MAE':>10s}")
for k, r in rows.items():
    L = r["latency"]
    print(f"{k:4s} {r['size_mb']:6.2f} {r['map50_95']:9.4f} {r['map50']:7.4f} {r['n_dets']:7d} "
          f"{L['t1']['mean_ms']:7.1f}±{L['t1']['std_ms']:<5.1f} {L[f't{THREADS[-1]}']['mean_ms']:7.1f}±{L[f't{THREADS[-1]}']['std_ms']:<5.1f} "
          f"{r['drift_50img']['box_mae_px']:8.3f} {r['drift_50img']['score_mae']:10.5f}")

yolov8n_op13_prep.onnx: 500 images in 35s, 69761 dets
[results] ablation.A = {'desc': 'FP32 (opset13, pre-processed)', 'path': 'yolov8n_op13_prep.onnx', 'size_mb': 12.24, 'map50_95': 0.38119653652033014, 'map50': 0.5137662020270171, 'n_dets': 69761, 'latency': {'t1': {'mean_ms': 93.31541458001993, 'std_ms': 1.9146632001960018}, 't4': {'mean_ms': 50.75060811997446, 'std_ms': 2.225033650939426}}, 'drift_50img': {'box_mae_px': 0.0, 'score_mae': 0.0}}  (Q12)
n_B_minmax.onnx: 500 images in 33s, 0 dets
[results] ablation.B = {'desc': 'INT8 MinMax, all nodes', 'path': 'n_B_minmax.onnx', 'size_mb': 3.43, 'map50_95': 0.0, 'map50': 0.0, 'n_dets': 0, 'latency': {'t1': {'mean_ms': 84.31511987999784, 'std_ms': 2.404835867691723}, 't4': {'mean_ms': 55.72233429996686, 'std_ms': 7.365053060612863}}, 'drift_50img': {'box_mae_px': 7.864907264709473, 'score_mae': 5.6156637583626434e-05}}  (Q12)
n_E1_minmax.onnx: 500 images in 36s, 65182 dets
[results] ablation.E1 = {'desc': 'INT8 MinMax, final Concat in 

In [15]:
# Q13 — (a) all-node collapse is method-independent? (b) E1's box-branch scale (c) chunked-vs-single Entropy under E2
def dq_scale(model_path, tensor):
    m = onnx.load(model_path)
    init = {i.name: onnx.numpy_helper.to_array(i) for i in m.graph.initializer}
    prod = {o: n for n in m.graph.node for o in n.output}
    n = prod.get(tensor)
    return float(init[n.input[1]]) if n is not None and n.op_type == "DequantizeLinear" else None

max_score = float(diag["fp32_score_range"][1])

# (a) output0 scale of the ALL-NODE models for each calibration method (40-image calibration, from Q9)
collapse = {}
for meth in ["MinMax", "Entropy", "Percentile"]:
    s = dq_scale(f"{TMP}/{meth}_single.onnx", "output0")
    collapse[meth] = {"output0_scale": s, "half_step": s / 2, "fp32_max_score": max_score,
                      "all_scores_round_to_zero": s / 2 > max_score}
    print(f"(a) {meth:10s} output0 scale {s:.4f} -> half-step {s/2:.3f} vs max FP32 score {max_score:.3f} "
          f"-> collapse: {s/2 > max_score}")
record("quant.all_node_collapse_by_method", collapse, "Q13")

# (b) in E1, what scales do the two inputs of the (FP32) final Concat carry?
m_e1 = onnx.load(QPATH["E1"])
cat = next(n for n in m_e1.graph.node if n.name == "/model.22/Concat_3")
e1_inputs = {}
for t in cat.input:
    e1_inputs[t] = dq_scale(QPATH["E1"], t)
    print(f"(b) E1 Concat_3 input {t}: DQ scale = {e1_inputs[t]}")
record("quant.e1_concat_input_scales", e1_inputs, "Q13")

# (c) chunked vs single Entropy, both with the E2 exclusion, both on the same 40 images
ent = {}
for mode, k in [("single", None), ("chunk10", 10)]:
    CHUNK["k"], CHUNK["log"] = k, []
    p = f"{TMP}/Entropy_E2_{mode}.onnx"
    quantize(CalibrationMethod.Entropy, YoloCalibReader(sub40), p, nodes_to_exclude=EXCL["E2"])
    ent[mode] = coco_map(run_eval(p, eval_ids, **EVAL_CFG)["int_pad"], eval_ids)["map50_95"]
CHUNK["k"] = None
delta = ent["chunk10"] - ent["single"]
accept = abs(delta) <= 0.005
print(f"(c) Entropy+E2 (40 img): single {ent['single']:.4f}  chunked {ent['chunk10']:.4f}  "
      f"delta {delta:+.4f} -> chunked Entropy {'ACCEPTED' if accept else 'REJECTED'} (rule |delta| <= 0.005)")
record("quant.entropy_chunk_check_E2_40img", {"single": ent["single"], "chunk10": ent["chunk10"],
       "delta": delta, "rule_abs_delta_le": 0.005, "accepted": accept}, "Q13")

(a) MinMax     output0 scale 2.5407 -> half-step 1.270 vs max FP32 score 0.930 -> collapse: True
(a) Entropy    output0 scale 2.5407 -> half-step 1.270 vs max FP32 score 0.930 -> collapse: True
(a) Percentile output0 scale 2.4973 -> half-step 1.249 vs max FP32 score 0.930 -> collapse: True
[results] quant.all_node_collapse_by_method = {'MinMax': {'output0_scale': 2.5406906604766846, 'half_step': 1.2703453302383423, 'fp32_max_score': 0.9295634031295776, 'all_scores_round_to_zero': True}, 'Entropy': {'output0_scale': 2.5406906604766846, 'half_step': 1.2703453302383423, 'fp32_max_score': 0.9295634031295776, 'all_scores_round_to_zero': True}, 'Percentile': {'output0_scale': 2.4972705841064453, 'half_step': 1.2486352920532227, 'fp32_max_score': 0.9295634031295776, 'all_scores_round_to_zero': True}}  (Q13)
(b) E1 Concat_3 input /model.22/Mul_2_output_0_DequantizeLinear_Output: DQ scale = 2.568486213684082
(b) E1 Concat_3 input /model.22/Sigmoid_output_0_DequantizeLinear_Output: DQ scale = 0.

In [16]:
# Q14 — C = Entropy+E2, D = Percentile+E2, E2_RR = MinMax+E2+reduce_range; then the full ablation table
def quantize(method, reader, out_path, nodes_to_exclude=None, reduce_range=False):
    t0 = time.time()
    quantize_static(N_PREP, out_path, calibration_data_reader=reader, quant_format=QuantFormat.QDQ,
                    per_channel=True, reduce_range=reduce_range,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8,
                    calibrate_method=method, nodes_to_exclude=nodes_to_exclude or [])
    return time.time() - t0

NEW = {
    "C_E2":  (CalibrationMethod.Entropy,    False, "INT8 Entropy (chunked, validated Q13c), E2 tail FP32"),
    "D_E2":  (CalibrationMethod.Percentile, False, "INT8 Percentile 99.999, E2 tail FP32"),
    "E2_RR": (CalibrationMethod.MinMax,     True,  "INT8 MinMax, E2 tail FP32, reduce_range (7-bit weights)"),
}
CHUNK["k"] = 25
for name, (meth, rr, desc) in NEW.items():
    QPATH[name] = f"{QDIR}/n_{name}.onnx"
    CHUNK["log"] = []
    secs = quantize(meth, YoloCalibReader(calib_ids), QPATH[name], nodes_to_exclude=EXCL["E2"], reduce_range=rr)
    print(f"built {name} in {secs:.0f}s ({CHUNK['log'][-1][1]} chunks of 25)")
    rows[name] = evaluate_config(name, QPATH[name], desc, "Q14")
CHUNK["k"] = None

record("ablation_notes", {"C_D_all_nodes_not_run": "Q13a: output0 half-step 1.25-1.27 > max score 0.93 for "
       "MinMax/Entropy/Percentile, so all-node C/D collapse like B; C/D evaluated with the E2 exclusion instead"}, "Q14")

# ── full table, generated from results.json ──
AB = json.load(open(RESULTS))["ablation"]
order = ["A", "B", "E1", "E2", "C_E2", "D_E2", "E2_RR"]
a = AB["A"]["value"]; tmax = f"t{THREADS[-1]}"
print(f"\n{'cfg':6s} {'MB':>6s} {'mAP50-95':>9s} {'Δ':>7s} {'mAP50':>7s} {'t1 ms':>8s} {'x':>5s} {tmax+' ms':>8s} {'x':>5s} {'boxMAE':>7s}  desc")
for k in order:
    r = AB[k]["value"]
    print(f"{k:6s} {r['size_mb']:6.2f} {r['map50_95']:9.4f} {r['map50_95'] - a['map50_95']:+7.4f} {r['map50']:7.4f} "
          f"{r['latency']['t1']['mean_ms']:8.1f} {a['latency']['t1']['mean_ms'] / r['latency']['t1']['mean_ms']:5.2f} "
          f"{r['latency'][tmax]['mean_ms']:8.1f} {a['latency'][tmax]['mean_ms'] / r['latency'][tmax]['mean_ms']:5.2f} "
          f"{r['drift_50img']['box_mae_px']:7.3f}  {r['desc']}")

best = max((k for k in order if k != "A"), key=lambda k: AB[k]["value"]["map50_95"])
b = AB[best]["value"]
record("quant.best_int8", {"config": best, "path": b["path"], "map50_95": b["map50_95"],
       "drop_vs_fp32": b["map50_95"] - a["map50_95"], "size_mb": b["size_mb"],
       "size_ratio_vs_fp32": a["size_mb"] / b["size_mb"]}, "Q14")

Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 242
Number of histogram bins : 128 (The number may increase depends on the data it collects)
Number of quantized bins : 128
built C_E2 in 263s (12 chunks of 25)
n_C_E2.onnx: 500 images in 34s, 74264 dets
[results] ablation.C_E2 = {'desc': 'INT8 Entropy (chunked, validated Q13c), E2 tail FP32', 'path': 'n_C_E2.onnx', 'size_mb': 3.49, 'map50_95': 0.34214490844534706, 'map50': 0.4682923264485575, 'n_dets': 74264, 'latency': {'t1': {'mean_ms': 80.96661533998031, 'std_ms': 2.318260107182767}, 't4': {'mean_ms': 53.468171799977426, 'std_ms': 8.183100745881003}}, 'drift_50img': {'box_mae_px': 6.388725757598877, 'score_mae': 2.2060061382944696e-05}}  (Q14)
Finding optimal threshold for each tensor using 'percentile' algorithm ...
Number of tensors : 242
Number of histogram bins : 2048
Percentile : (0.0010000000000047748,99.999)
built D_E2 in 220s (12 chunks of 25)
n_D_E2.onnx: 500 images in 37s, 75653 de

In [17]:
print("session alive:", all(n in globals() for n in ["QPATH", "N_PREP", "eval_ids", "calib_ids", "run_eval", "coco_map", "make_sess"]))

session alive: True


In [18]:
# D1 — Qualcomm AI Hub: install, configure from Kaggle Secret (token never printed), inspect API, list devices
!pip install -q qai-hub
import subprocess
from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("QAI_HUB_API_TOKEN")
assert TOKEN and len(TOKEN) > 10, "secret empty — check Add-ons → Secrets is attached to this notebook"
redact = lambda s: s.replace(TOKEN, "***")

r = subprocess.run(["qai-hub", "configure", "--api_token", TOKEN], capture_output=True, text=True)
print("configure exit code:", r.returncode)
print(redact(r.stdout)[-400:], redact(r.stderr)[-400:])

import qai_hub as hub
print("qai-hub", pkg_version("qai-hub"))
print("public API:", [n for n in dir(hub) if not n.startswith("_")])

devs = hub.get_devices()
print(f"\n{len(devs)} devices visible. Snapdragon 8 Gen 3 / Galaxy S24 candidates:")
cands = [d for d in devs if "S24" in d.name or "8gen3" in " ".join(d.attributes).replace(" ", "").lower()]
for d in (cands or devs)[:15]:
    print(f"  {d.name} | {d.os} | {[a for a in d.attributes if a.startswith(('chipset', 'framework', 'format'))]}")
record("aihub.client", {"qai_hub_version": pkg_version("qai-hub"), "n_devices": len(devs),
                        "n_8gen3_candidates": len(cands)}, "D1")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.41.5 requires s3transfer<0.16.0,>=0.15.0, but you have s3transfer 0.13.1 which is incompatible.
configure exit code: 0
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = ***
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli


 2026-10-11 21:49:05.288 - INFO - Enabling verbose logging.

qai-hub 0.56.0
public API: ['Client', 'ClientConfig', 'CompileJob', 'CompileJobResult', 'CompileJobSummary', 'Dataset', 'Device', 'Error', 'Framework', 'IOSpec', 'InferenceJob', 'InferenceJobResult', 'In

In [19]:
# D2 — locate models, clean for AI Hub (unused opset domains + value_info on graph IO), submit/re-fetch compile jobs
import inspect
print("submit_compile_job", inspect.signature(hub.submit_compile_job))

def locate(name):
    for root in (WORK, "/kaggle/input"):
        for r, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in ("train2017", "test2017", "unlabeled2017", "val2017")]
            if name in files:
                return os.path.join(r, name)
    raise FileNotFoundError(f"{name} — Run All, or attach NB3's committed output")

def clean_for_aihub(src, dst):
    """Strict-ONNX cleanup: drop unused opset imports and value_info entries that duplicate graph inputs/outputs."""
    m = onnx.load(src)
    used = {n.domain for n in m.graph.node}
    keep = [o for o in m.opset_import if o.domain in used or o.domain in ("", "ai.onnx")]
    removed_domains = sorted({o.domain for o in m.opset_import} - {o.domain for o in keep})
    del m.opset_import[:]
    m.opset_import.extend(keep)

    io_names = {t.name for t in m.graph.input} | {t.name for t in m.graph.output}
    vi_keep = [v for v in m.graph.value_info if v.name not in io_names]
    removed_vi = sorted(v.name for v in m.graph.value_info if v.name in io_names)
    del m.graph.value_info[:]
    m.graph.value_info.extend(vi_keep)

    onnx.checker.check_model(m, full_check=True)
    onnx.save(m, dst)
    x = np.random.default_rng(0).random((1, 3, 640, 640), dtype=np.float32)
    y0 = ort.InferenceSession(src, providers=["CPUExecutionProvider"]).run(None, {"images": x})[0]
    y1 = ort.InferenceSession(dst, providers=["CPUExecutionProvider"]).run(None, {"images": x})[0]
    assert np.array_equal(y0, y1), f"cleanup changed outputs for {src}"
    return {"removed_domains": removed_domains, "removed_io_value_info": removed_vi}

SRC = {"fp32": locate("yolov8n_op13_prep.onnx"), "int8_E2": locate("n_E2_minmax.onnx"),
       "int8_E2_RR": locate("n_E2_RR.onnx")}
print("sources:", SRC)

DEVICE_NAME = "Samsung Galaxy S24 (Family)"
DEVICE = hub.Device(DEVICE_NAME)
TARGET = "--target_runtime qnn_dlc"
JOB_IDS = {"compile_fp32": "jgd688vkp", "compile_int8_E2": "j57okkdqg", "compile_int8_E2_RR": "jp4emmwqg"}
compile_jobs, cleaned = {}, {}
for k, src in SRC.items():
    dst = f"{WORK}/aihub_{k}.onnx"
    cleaned[k] = clean_for_aihub(src, dst)            # always run: cheap, and keeps the evidence in results.json
    key = f"compile_{k}"
    if key in JOB_IDS:
        compile_jobs[k] = hub.get_job(JOB_IDS[key])
        print(f"{k}: cleaned {cleaned[k]}; re-fetched compile job {JOB_IDS[key]}")
        continue
    print(f"{k}: cleaned {cleaned[k]} (outputs bit-identical) -> submitting")
    compile_jobs[k] = hub.submit_compile_job(model=dst, device=DEVICE, name=f"yolov8n_{k}",
                                             input_specs={"images": ((1, 3, 640, 640), "float32")},
                                             options=TARGET)

print("\nwaiting for compile jobs (a few minutes each)...")
status, ok = {}, {}
for k, j in compile_jobs.items():
    st = j.wait()
    ok[k] = getattr(st, "success", "SUCCESS" in str(st).upper())
    status[k] = {"job_id": j.job_id, "success": bool(ok[k]), "status": str(st).replace("\n", " | ")}
    print(f"{k:11s} {j.job_id}  success={ok[k]}")
    if not ok[k]:
        print("   ", str(st).replace("\n", " | "))

if all(ok.values()):
    print("\nAll compiled. Paste into JOB_IDS:", json.dumps({f"compile_{k}": j.job_id for k, j in compile_jobs.items()}))
record("aihub.device", {"name": DEVICE_NAME, "chipset": "qualcomm-snapdragon-8gen3 (sm8650)", "os": "Android 14"}, "D2")
record("aihub.compile", {"target_runtime": TARGET, "jobs": status, "cleanup": cleaned}, "D2")

submit_compile_job (model: 'Model | SourceModel | str | Path | None', device: 'Device | list[Device]', name: 'str | None' = None, input_specs: 'InputSpecs | None' = None, options: 'str' = '', single_compile: 'bool' = True, calibration_data: 'Dataset | DatasetEntries | str | None' = None, retry: 'bool' = True, project: 'str | Project | None' = None) -> 'CompileJob | list[CompileJob]'
sources: {'fp32': '/kaggle/working/yolov8n_op13_prep.onnx', 'int8_E2': '/kaggle/working/int8/n_E2_minmax.onnx', 'int8_E2_RR': '/kaggle/working/int8/n_E2_RR.onnx'}
fp32: cleaned {'removed_domains': ['ai.onnx.ml', 'ai.onnx.preview.training', 'ai.onnx.training', 'com.microsoft', 'com.microsoft.experimental', 'com.microsoft.nchwc', 'org.pytorch.aten'], 'removed_io_value_info': ['output0']}; re-fetched compile job jgd688vkp
int8_E2: cleaned {'removed_domains': ['ai.onnx.ml', 'ai.onnx.preview.training', 'ai.onnx.training', 'com.microsoft', 'com.microsoft.experimental', 'com.microsoft.nchwc', 'org.pytorch.aten'], 

In [20]:
# D3 — on-device profiling on the S24; print the profile schema, then extract time / memory / compute units / CPU fallbacks
from collections import Counter
PROFILE_IDS = {"profile_fp32": "jp0odd12p", "profile_int8_E2": "jp8j663z5", "profile_int8_E2_RR": "jgk6ooly5"}

profile_jobs = {}
for k, cj in compile_jobs.items():
    key = f"profile_{k}"
    if key in PROFILE_IDS:
        profile_jobs[k] = hub.get_job(PROFILE_IDS[key]); print(f"{k}: re-fetched {PROFILE_IDS[key]}")
    else:
        profile_jobs[k] = hub.submit_profile_job(model=cj.get_target_model(), device=DEVICE, name=f"yolov8n_{k}_profile")

prof, ok = {}, {}
for k, pj in profile_jobs.items():
    st = pj.wait()
    ok[k] = getattr(st, "success", "SUCCESS" in str(st).upper())
    print(f"{k:11s} {pj.job_id} success={ok[k]}")
    if ok[k]:
        prof[k] = pj.download_profile()
    else:
        print("   ", str(st).replace("\n", " | "))

# ── schema first (don't assume it) ──
p0 = next(iter(prof.values()))
print("\nprofile top-level keys:", list(p0.keys()))
for key, val in p0.items():
    if isinstance(val, dict):
        print(f"  {key}: {list(val.keys())}")
    elif isinstance(val, list) and val and isinstance(val[0], dict):
        print(f"  {key}: list[{len(val)}] of {list(val[0].keys())}")

# ── extract ──
summary = {}
try:
    for k, p in prof.items():
        es, det = p["execution_summary"], p.get("execution_detail", [])
        summary[k] = {
            "inference_ms": es["estimated_inference_time"] / 1000.0,            # reported in microseconds
            "peak_memory_mb": es.get("estimated_inference_peak_memory", float("nan")) / 2**20,
            "layers_total": len(det),
            "layers_per_unit": dict(Counter(l.get("compute_unit") for l in det)),
            "cpu_fallback_ops": dict(Counter(l.get("type") for l in det if l.get("compute_unit") == "CPU")),
            "execution_summary_raw": {kk: vv for kk, vv in es.items() if isinstance(vv, (int, float, str))},
        }
    print(f"\n{'model':11s} {'ms':>8s} {'peak MB':>8s}  layers per unit           CPU fallback ops")
    for k, s in summary.items():
        print(f"{k:11s} {s['inference_ms']:8.2f} {s['peak_memory_mb']:8.1f}  {str(s['layers_per_unit']):24s}  {s['cpu_fallback_ops'] or 'none'}")
    record("aihub.profile", summary, "D3")
except KeyError as e:
    print(f"\nKeyError {e}: the schema differs from what I assumed — paste the key listing above")

if all(ok.values()):
    print("\nPaste into PROFILE_IDS:", json.dumps({f"profile_{k}": j.job_id for k, j in profile_jobs.items()}))

fp32: re-fetched jp0odd12p
int8_E2: re-fetched jp8j663z5
int8_E2_RR: re-fetched jgk6ooly5
fp32        jp0odd12p success=True
int8_E2     jp8j663z5 success=True
int8_E2_RR  jgk6ooly5 success=True

profile top-level keys: ['execution_summary', 'execution_detail']
  execution_summary: ['estimated_inference_time', 'estimated_inference_peak_memory', 'first_load_time', 'first_load_peak_memory', 'warm_load_time', 'warm_load_peak_memory', 'compile_time', 'compile_peak_memory', 'compile_memory_increase_range', 'compile_memory_peak_range', 'first_load_memory_increase_range', 'first_load_memory_peak_range', 'warm_load_memory_increase_range', 'warm_load_memory_peak_range', 'inference_memory_increase_range', 'inference_memory_peak_range', 'all_compile_times', 'all_first_load_times', 'all_warm_load_times', 'all_inference_times']
  execution_detail: list[247] of ['name', 'type', 'compute_unit', 'execution_time', 'execution_cycles']

model             ms  peak MB  layers per unit           CPU fallbac

In [21]:
# D4 — on-device inference parity: 10 eval images on the S24, raw outputs decoded + NMS'd locally, vs local ORT
INFER_IDS = {}   # after success, paste the printed dict here

par_ids = eval_ids[:10]
xs = []
for iid in par_ids:
    img = cv2.imread(f"{COCO_IMG_DIR}/{coco_gt.loadImgs(iid)[0]['file_name']}")
    xs.append(to_tensor(letterbox(img)[0]))

inf_jobs = {}
for k in ["fp32", "int8_E2", "int8_E2_RR"]:
    key = f"infer_{k}"
    if key in INFER_IDS:
        inf_jobs[k] = hub.get_job(INFER_IDS[key]); print(f"{k}: re-fetched {INFER_IDS[key]}")
    else:
        inf_jobs[k] = hub.submit_inference_job(model=compile_jobs[k].get_target_model(), device=DEVICE,
                                               inputs={"images": xs}, name=f"yolov8n_{k}_parity10")

dev = {}
for k, j in inf_jobs.items():
    j.wait()
    out = j.download_output_data()
    print(f"{k}: device output keys {list(out.keys())}, first shape {np.asarray(list(out.values())[0][0]).shape}")
    dev[k] = [np.asarray(a, dtype=np.float32).reshape(1, 84, -1) for a in list(out.values())[0]]

def iou_xyxy(a, b):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih; uni = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / uni if uni > 0 else 0.0

def det_match(raw_a, raw_b, conf=0.25, iou=0.45):
    """Demo-setting dets from two raw outputs; greedy match by class + IoU>0.5. Returns matched, n_b, score drift."""
    ba, ca, ka = decode_nms(raw_a, conf, iou, 300)
    bb, cb, kb = decode_nms(raw_b, conf, iou, 300)
    used, matched, drift = set(), 0, []
    for i in range(len(ca)):
        best, bj = 0.5, None
        for j in range(len(cb)):
            if j in used or kb[j] != ka[i]:
                continue
            v = iou_xyxy(ba[i], bb[j])
            if v > best:
                best, bj = v, j
        if bj is not None:
            used.add(bj); matched += 1; drift.append(abs(float(ca[i]) - float(cb[bj])))
    return matched, len(cb), drift

def compare(dev_raws, ref_path):
    s = make_sess(ref_path)
    box, score, m_tot, n_tot, drift = [], [], 0, 0, []
    for x, yd in zip(xs, dev_raws):
        yr = s.run(None, {"images": x})[0]
        box.append(np.abs(yd[:, :4] - yr[:, :4]).mean()); score.append(np.abs(yd[:, 4:] - yr[:, 4:]).mean())
        m, n, d = det_match(yd, yr); m_tot += m; n_tot += n; drift += d
    return {"raw_box_mae_px": float(np.mean(box)), "raw_score_mae": float(np.mean(score)),
            "dets_matched": m_tot, "dets_ref": n_tot, "match_pct": 100.0 * m_tot / max(n_tot, 1),
            "mean_score_drift": float(np.mean(drift)) if drift else None}

parity = {
    "fp32_device_vs_fp32_local":       compare(dev["fp32"],       SRC["fp32"]),
    "int8E2_device_vs_int8E2_local":   compare(dev["int8_E2"],    SRC["int8_E2"]),
    "int8E2_device_vs_fp32_local":     compare(dev["int8_E2"],    SRC["fp32"]),
    "int8E2RR_device_vs_fp32_local":   compare(dev["int8_E2_RR"], SRC["fp32"]),
}
for k, v in parity.items():
    print(f"{k:32s} {v}")
record("aihub.device_parity_10img", parity, "D4")
print("\nPaste into INFER_IDS:", json.dumps({f"infer_{k}": j.job_id for k, j in inf_jobs.items()}))

Uploading dataset: 100%|██████████| 9.71M/9.71M [00:00<00:00, 13.3MB/s]


Scheduled inference job (jp2o44rmg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp2o44rmg/



Uploading dataset: 100%|██████████| 9.71M/9.71M [00:00<00:00, 12.8MB/s]


Scheduled inference job (jpy8qqo4g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpy8qqo4g/



Uploading dataset: 100%|██████████| 9.71M/9.71M [00:00<00:00, 13.7MB/s]


...

fp32: device output keys ['output_0'], first shape (1, 84, 8400)
Waiting for inference job (jpy8qqo4g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmp0gf91t_q.h5: 100%|██████████| 881k/881k [00:00<00:00, 1.71MB/s]


int8_E2: device output keys ['output_0'], first shape (1, 84, 8400)
Waiting for inference job (jp0oddmep) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmphz74mymm.h5: 100%|██████████| 878k/878k [00:00<00:00, 1.40MB/s]


int8_E2_RR: device output keys ['output_0'], first shape (1, 84, 8400)
fp32_device_vs_fp32_local        {'raw_box_mae_px': 0.4440409243106842, 'raw_score_mae': 7.308255590032786e-06, 'dets_matched': 84, 'dets_ref': 84, 'match_pct': 100.0, 'mean_score_drift': 0.005827471613883972}
int8E2_device_vs_int8E2_local    {'raw_box_mae_px': 4.450164794921875, 'raw_score_mae': 2.698513890209142e-05, 'dets_matched': 70, 'dets_ref': 81, 'match_pct': 86.41975308641975, 'mean_score_drift': 0.052673496944563726}
int8E2_device_vs_fp32_local      {'raw_box_mae_px': 6.06488037109375, 'raw_score_mae': 2.4712609956623055e-05, 'dets_matched': 76, 'dets_ref': 84, 'match_pct': 90.47619047619048, 'mean_score_drift': 0.057607258621015046}
int8E2RR_device_vs_fp32_local    {'raw_box_mae_px': 6.324544429779053, 'raw_score_mae': 2.654331001394894e-05, 'dets_matched': 74, 'dets_ref': 84, 'match_pct': 88.0952380952381, 'mean_score_drift': 0.07253883336041425}
[results] aihub.device_parity_10img = {'fp32_device_vs_fp3

In [22]:
# D5 — Kaggle CPU vs Snapdragon 8 Gen 3 NPU: latency, size, mAP (mAP = local ORT eval of the same ONNX file)
import pandas as pd
R = json.load(open(RESULTS))
ab = {k: R["ablation"][k]["value"] for k in ["A", "E2", "E2_RR"]}
pf = R["aihub"]["profile"]["value"]
tmx = [t for t in ab["A"]["latency"] if t != "t1"][0]
comp = []
for label, abk, pk in [("FP32", "A", "fp32"), ("INT8 E2", "E2", "int8_E2"), ("INT8 E2_RR", "E2_RR", "int8_E2_RR")]:
    r = ab[abk]
    comp.append({"model": label, "onnx_mb": r["size_mb"],
                 "mAP50-95 (local eval)": round(r["map50_95"], 4), "mAP50 (local eval)": round(r["map50"], 4),
                 "Kaggle CPU t1 ms": round(r["latency"]["t1"]["mean_ms"], 1),
                 f"Kaggle CPU {tmx} ms": round(r["latency"][tmx]["mean_ms"], 1),
                 "S24 NPU ms": round(pf[pk]["inference_ms"], 2),
                 "layers on NPU": f"{pf[pk]['layers_per_unit'].get('NPU', 0)}/{pf[pk]['layers_total']}",
                 "CPU fallback layers": sum(pf[pk]["cpu_fallback_ops"].values())})
display(pd.DataFrame(comp))
summ = {"rows": comp,
        "npu_int8_speedup_E2": pf["fp32"]["inference_ms"] / pf["int8_E2"]["inference_ms"],
        "cpu_int8_speedup_t1_E2": ab["A"]["latency"]["t1"]["mean_ms"] / ab["E2"]["latency"]["t1"]["mean_ms"],
        f"cpu_int8_speedup_{tmx}_E2": ab["A"]["latency"][tmx]["mean_ms"] / ab["E2"]["latency"][tmx]["mean_ms"],
        "note": "mAP is from local ORT eval of the same ONNX file; on-device mAP not measured (10-image parity only, D4). "
                "CPU and NPU latencies are different hardware classes — compare FP32 vs INT8 within a row group, not across."}
print({k: v for k, v in summ.items() if k != "rows"})
record("aihub.comparison", summ, "D5")

,model,onnx_mb,mAP50-95 (local eval),mAP50 (local eval),Kaggle CPU t1 ms,Kaggle CPU t4 ms,S24 NPU ms,layers on NPU,CPU fallback layers
0,FP32,12.24,0.3812,0.5138,93.3,50.8,3.34,247/247,0
1,INT8 E2,3.49,0.3368,0.4625,81.7,51.9,2.19,251/251,0
2,INT8 E2_RR,3.49,0.3663,0.4999,80.8,52.7,2.20,251/251,0


{'npu_int8_speedup_E2': 1.5239616613418532, 'cpu_int8_speedup_t1_E2': 1.1428224957746245, 'cpu_int8_speedup_t4_E2': 0.978158871136885, 'note': 'mAP is from local ORT eval of the same ONNX file; on-device mAP not measured (10-image parity only, D4). CPU and NPU latencies are different hardware classes — compare FP32 vs INT8 within a row group, not across.'}
[results] aihub.comparison = {'rows': [{'model': 'FP32', 'onnx_mb': 12.24, 'mAP50-95 (local eval)': 0.3812, 'mAP50 (local eval)': 0.5138, 'Kaggle CPU t1 ms': 93.3, 'Kaggle CPU t4 ms': 50.8, 'S24 NPU ms': 3.34, 'layers on NPU': '247/247', 'CPU fallback layers': 0}, {'model': 'INT8 E2', 'onnx_mb': 3.49, 'mAP50-95 (local eval)': 0.3368, 'mAP50 (local eval)': 0.4625, 'Kaggle CPU t1 ms': 81.7, 'Kaggle CPU t4 ms': 51.9, 'S24 NPU ms': 2.19, 'layers on NPU': '251/251', 'CPU fallback layers': 0}, {'model': 'INT8 E2_RR', 'onnx_mb': 3.49, 'mAP50-95 (local eval)': 0.3663, 'mAP50 (local eval)': 0.4999, 'Kaggle CPU t1 ms': 80.8, 'Kaggle CPU t4 ms'

In [23]:
# R0 — render helpers; everything below is read from results.json
from IPython.display import Markdown, display
R = json.load(open(RESULTS))

def rv(rec, *keys):
    node = R
    for p in rec.split("."):
        node = node[p]
    v = node["value"]
    for k in keys:
        v = v[k]
    return v, node["cell"]

def md(s):
    display(Markdown(s))

AB = {k: rv(f"ablation.{k}")[0] for k in ["A", "B", "E1", "E2", "C_E2", "D_E2", "E2_RR"]}
ABC = {k: rv(f"ablation.{k}")[1] for k in AB}
tmax = [t for t in AB["A"]["latency"] if t != "t1"][0]
best, c_best = rv("quant.best_int8")
dg, c_dg = rv("quant.full_int8_output_diagnosis")
col, c_col = rv("quant.all_node_collapse_by_method")
e1s, c_e1 = rv("quant.e1_concat_input_scales")
e1_box = next(v for k, v in e1s.items() if "Mul_2" in k)
e1_score = next(v for k, v in e1s.items() if "Sigmoid" in k)
entc, c_ent = rv("quant.entropy_chunk_check_E2_40img")
moved, c_mv = rv("quant.entropy_chunk_moved_tensors_40img")
mem, c_mem = rv("quant.calib_memory_estimate")
chk, c_chk = rv("quant.chunked_calibration_check_40img")
op13, c_op = rv("quant.opset13_conversion_n")
sub, c_sub = rv("subsets")
hp, c_hp = rv("map.halfpixel_ablation_n")
fp = {v: rv(f"map.fp32.{v}")[0] for v in "nsm"}
latf, c_lat = rv("latency.fp32")
prof, c_prof = rv("aihub.profile")
par, c_par = rv("aihub.device_parity_10img")
devc, c_dev = rv("aihub.device")
cmpd, c_cmp = rv("aihub.comparison")
cpu, c_cpu = rv("env.cpu_model"); topo, _ = rv("env.topology"); isa, _ = rv("env.isa"); ver, _ = rv("env.versions")
dA = lambda k: AB[k]["map50_95"] - AB["A"]["map50_95"]
relA = lambda k: 100 * dA(k) / AB["A"]["map50_95"]
cpux = lambda k, t: AB["A"]["latency"][t]["mean_ms"] / AB[k]["latency"][t]["mean_ms"]
npux = prof["fp32"]["inference_ms"] / prof["int8_E2"]["inference_ms"]
print("helpers ready")

helpers ready


In [24]:
# R1 — Results card
L = lambda k, t: AB[k]["latency"][t]["mean_ms"]
md(f"""## Results card
Environment: {cpu} · {topo['Core(s) per socket']} cores × {topo['Thread(s) per core']} threads · AVX-512F {isa['avx512f']}, VNNI {isa['avx512_vnni']} · ORT {ver['onnxruntime']} · onnx {ver['onnx']} · device: {devc['name']} ({devc['chipset']}) — cells {c_cpu}, {c_dev}

| Metric | Value | Cell |
|---|---|---|
| FP32 mAP50-95, 500 COCO val images (n / s / m) | {fp['n']['map50_95']:.4f} / {fp['s']['map50_95']:.4f} / {fp['m']['map50_95']:.4f} | Q4, Q5 |
| Full INT8 (all nodes) | {AB['B']['n_dets']} detections, mAP {AB['B']['map50_95']:.4f}; output0 scale {dg['output_scale']:.3f} → every score (max {dg['fp32_score_range'][1]:.3f}) rounds to 0 | {c_dg}, {ABC['B']} |
| INT8, final Concat in FP32 (E1) | mAP {AB['E1']['map50_95']:.4f} ({relA('E1'):+.1f}%); box branch scale {e1_box:.2f} px/step | {ABC['E1']}, {c_e1} |
| INT8, decode tail in FP32 (E2) | mAP {AB['E2']['map50_95']:.4f} ({relA('E2'):+.1f}%) | {ABC['E2']} |
| Calibration: MinMax / Entropy / Percentile (E2) | {AB['E2']['map50_95']:.4f} / {AB['C_E2']['map50_95']:.4f} / {AB['D_E2']['map50_95']:.4f} | {ABC['E2']}, {ABC['C_E2']} |
| **Best INT8: E2 + reduce_range** | **mAP {best['map50_95']:.4f} vs FP32 {AB['A']['map50_95']:.4f} ({relA('E2_RR'):+.1f}%), {AB['A']['size_mb']:.2f} → {best['size_mb']:.2f} MB ({best['size_ratio_vs_fp32']:.1f}× smaller)** | {c_best} |
| INT8 speed on Kaggle CPU (E2_RR vs FP32) | t1 {cpux('E2_RR','t1'):.2f}×, {tmax} {cpux('E2_RR',tmax):.2f}× (no VNNI) | {ABC['E2_RR']} |
| **Snapdragon 8 Gen 3 NPU latency (FP32 → INT8 E2)** | **{prof['fp32']['inference_ms']:.2f} → {prof['int8_E2']['inference_ms']:.2f} ms ({npux:.2f}×)** | {c_prof} |
| Layers on NPU / CPU fallback | FP32 {prof['fp32']['layers_per_unit'].get('NPU',0)}/{prof['fp32']['layers_total']}, INT8 {prof['int8_E2']['layers_per_unit'].get('NPU',0)}/{prof['int8_E2']['layers_total']}; fallback ops: {prof['int8_E2']['cpu_fallback_ops'] or 'none'} | {c_prof} |
| On-device parity, 10 images | FP32 device vs local {par['fp32_device_vs_fp32_local']['match_pct']:.0f}% matched; INT8 device vs FP32 local {par['int8E2_device_vs_fp32_local']['match_pct']:.0f}% | {c_par} |
| Calibration RAM, naive vs chunked | {mem['gb_for_all_calib']} GB needed vs {mem['ram_total_gb']} GB RAM → chunked; MinMax chunked == single-pass (max rel diff {chk['MinMax']['max_rel_diff']}) | {c_mem}, {c_chk} |

Full ablation table: Q14. CPU vs NPU table: D5.
""")

## Results card
Environment: Intel(R) Xeon(R) CPU @ 2.00GHz · 2 cores × 2 threads · AVX-512F True, VNNI False · ORT 1.27.0 · onnx 1.22.0 · device: Samsung Galaxy S24 (Family) (qualcomm-snapdragon-8gen3 (sm8650)) — cells Q1, D2

| Metric | Value | Cell |
|---|---|---|
| FP32 mAP50-95, 500 COCO val images (n / s / m) | 0.3812 / 0.4629 / 0.5117 | Q4, Q5 |
| Full INT8 (all nodes) | 0 detections, mAP 0.0000; output0 scale 2.541 → every score (max 0.930) rounds to 0 | Q10, Q12 |
| INT8, final Concat in FP32 (E1) | mAP 0.2835 (-25.6%); box branch scale 2.57 px/step | Q12, Q13 |
| INT8, decode tail in FP32 (E2) | mAP 0.3368 (-11.6%) | Q12 |
| Calibration: MinMax / Entropy / Percentile (E2) | 0.3368 / 0.3421 / 0.3428 | Q12, Q14 |
| **Best INT8: E2 + reduce_range** | **mAP 0.3663 vs FP32 0.3812 (-3.9%), 12.24 → 3.49 MB (3.5× smaller)** | Q14 |
| INT8 speed on Kaggle CPU (E2_RR vs FP32) | t1 1.16×, t4 0.96× (no VNNI) | Q14 |
| **Snapdragon 8 Gen 3 NPU latency (FP32 → INT8 E2)** | **3.34 → 2.19 ms (1.52×)** | D3 |
| Layers on NPU / CPU fallback | FP32 247/247, INT8 251/251; fallback ops: none | D3 |
| On-device parity, 10 images | FP32 device vs local 100% matched; INT8 device vs FP32 local 90% | D4 |
| Calibration RAM, naive vs chunked | 69.0 GB needed vs 31.3 GB RAM → chunked; MinMax chunked == single-pass (max rel diff 0.0) | Q8, Q9 |

Full ablation table: Q14. CPU vs NPU table: D5.


In [25]:
# R2 — Gotchas I hit (real ones only)
md(f"""## Gotchas I hit
1. **Calibration would need {mem['gb_for_all_calib']} GB of RAM.** Symptom: {mem['mb_per_image']} MB of activations per image × {sub['calib_images']} images vs {mem['ram_total_gb']} GB RAM ({c_mem}). Cause: ORT's calibrators keep every intermediate output of every image before computing ranges. Fix: chunked `collect_data`; MinMax chunked is bit-identical to single-pass (max rel diff {chk['MinMax']['max_rel_diff']}), Percentile median drift {chk['Percentile']['median_rel_diff']:.1e} ({c_chk}).
2. **`CalibMaxIntermediateOutputs` discards data in this ORT build.** Found by reading the installed source before using it (printed in Q8): when the buffer fills, MinMax calls `clear_collected_data()` without `compute_data()`. Not used.
3. **Per-channel INT8 model wouldn't load: `Unrecognized attribute: axis for operator DequantizeLinear`.** Cause: NB1 exported at opset 12; per-axis Q/DQ needs opset ≥ 13. Fix: `version_converter` 12 → 13; outputs identical (max diff {op13['max_score_diff']}) ({c_op}).
4. **Full INT8 produced {AB['B']['n_dets']} detections.** Cause: `output0` concatenates boxes ({dg['fp32_box_range'][0]:.1f}–{dg['fp32_box_range'][1]:.1f}) and scores (≤ {dg['fp32_score_range'][1]:.3f}) under one scale {dg['output_scale']:.3f}; every score rounds to 0 for all three calibration methods ({c_dg}, {c_col}). Fix: keep the decode tail in FP32 (E2).
5. **`coco_map` crashed on 0 detections** (`IndexError` in `pycocotools.loadRes`). Fix: an empty result list returns mAP = 0 (Q4).
6. **Chunked Entropy moved {moved['moved_gt_1pct']}/{moved['total']} activation scales, concentrated in `model.19/21/22`** ({c_mv}). Under E2 the mAP difference was {entc['delta']:+.4f} (rule |Δ| ≤ 0.005) → accepted ({c_ent}).
7. **Commit failed with `NameError` in Q10.** A helper defined in an earlier, replaced version of the cell still existed in interactive memory. Fix: Q10 defines everything it uses; every commit now starts from an empty kernel.
8. **AI Hub rejected the runtime `qnn_context_binary`.** qai-hub {R['aihub']['client']['value']['qai_hub_version']} accepts `onnx`, `precompiled_qnn_onnx`, `qnn_dlc`, `tflite`. Fix: `qnn_dlc` (D2).
9. **AI Hub compile failed: `Tensors {{'output0'}} occur in value_info but also in model IO`.** Cause: shape inference in `quant_pre_process` wrote `value_info` for the graph output (invalid per the ONNX IR spec; ORT tolerates it). Fix: strip IO entries from `value_info` and unused opset domains; outputs bit-identical (D2).
10. **`submit_profile_job` is deprecated** in this qai-hub (it returns an InferenceJob); job IDs are pinned so commits re-fetch instead of resubmitting (D3).
""")

## Gotchas I hit
1. **Calibration would need 69.0 GB of RAM.** Symptom: 235.5 MB of activations per image × 300 images vs 31.3 GB RAM (Q8). Cause: ORT's calibrators keep every intermediate output of every image before computing ranges. Fix: chunked `collect_data`; MinMax chunked is bit-identical to single-pass (max rel diff 0.0), Percentile median drift 6.3e-05 (Q9).
2. **`CalibMaxIntermediateOutputs` discards data in this ORT build.** Found by reading the installed source before using it (printed in Q8): when the buffer fills, MinMax calls `clear_collected_data()` without `compute_data()`. Not used.
3. **Per-channel INT8 model wouldn't load: `Unrecognized attribute: axis for operator DequantizeLinear`.** Cause: NB1 exported at opset 12; per-axis Q/DQ needs opset ≥ 13. Fix: `version_converter` 12 → 13; outputs identical (max diff 0.0) (Q6b).
4. **Full INT8 produced 0 detections.** Cause: `output0` concatenates boxes (2.6–637.1) and scores (≤ 0.930) under one scale 2.541; every score rounds to 0 for all three calibration methods (Q10, Q13). Fix: keep the decode tail in FP32 (E2).
5. **`coco_map` crashed on 0 detections** (`IndexError` in `pycocotools.loadRes`). Fix: an empty result list returns mAP = 0 (Q4).
6. **Chunked Entropy moved 83/242 activation scales, concentrated in `model.19/21/22`** (Q10). Under E2 the mAP difference was +0.0023 (rule |Δ| ≤ 0.005) → accepted (Q13).
7. **Commit failed with `NameError` in Q10.** A helper defined in an earlier, replaced version of the cell still existed in interactive memory. Fix: Q10 defines everything it uses; every commit now starts from an empty kernel.
8. **AI Hub rejected the runtime `qnn_context_binary`.** qai-hub 0.56.0 accepts `onnx`, `precompiled_qnn_onnx`, `qnn_dlc`, `tflite`. Fix: `qnn_dlc` (D2).
9. **AI Hub compile failed: `Tensors {'output0'} occur in value_info but also in model IO`.** Cause: shape inference in `quant_pre_process` wrote `value_info` for the graph output (invalid per the ONNX IR spec; ORT tolerates it). Fix: strip IO entries from `value_info` and unused opset domains; outputs bit-identical (D2).
10. **`submit_profile_job` is deprecated** in this qai-hub (it returns an InferenceJob); job IDs are pinned so commits re-fetch instead of resubmitting (D3).


In [26]:
# R3 — Limitations and what I'd do next
md(f"""## Limitations
- **Subset evaluation:** {sub['eval_images']} COCO val2017 images ({sub['eval_gt_boxes_non_crowd']} GT boxes), not full val. FP32 n/s/m land 0.8–1.4 points above Ultralytics' full-val numbers; ranking calibration methods within ~0.6 points is not reliable at this size.
- **Calibration data** comes from val2017 (disjoint from eval, overlap {sub['overlap']}), not train2017.
- **PTQ only**, no QAT. **reduce_range's gain is consistent with U8×S8 saturation on a non-VNNI CPU but not proven**; the device check (D4) is 10 images.
- **Kaggle CPU is not an edge device** ({cpu}, AVX-512 without VNNI). CPU and NPU latencies are different hardware classes; the comparison that matters is FP32 vs INT8 within each.
- **On-device accuracy:** mAP is from local ORT eval of the same ONNX. On the device, only 10-image detection parity was measured, and device INT8 matches local INT8 at {par['int8E2_device_vs_int8E2_local']['match_pct']:.0f}%, so device mAP could differ.
- **AI Hub numbers** are for one device ({devc['name']}); peak memory is process-level, not model-level; the FP32 model's internal precision on the NPU is not reported.
- **Box drift** (box MAE) averages over all 8,400 anchors, mostly background, so it overstates the error on boxes that matter.
- Half-pixel scale-back fix measured at {hp['delta']:+.4f} mAP50-95 ({c_hp}).

## What I'd do next
- On-device mAP over the 500 images (batched inference jobs), then a QAT run to close the remaining {abs(dA('E2_RR')):.3f} mAP.
- Split `output0` into separate box and score outputs at export, so the head can be quantized without the shared-scale collapse.
- Mixed-precision search (only the sensitive layers in FP16), and the same study on yolov8s/m.
""")

## Limitations
- **Subset evaluation:** 500 COCO val2017 images (3456 GT boxes), not full val. FP32 n/s/m land 0.8–1.4 points above Ultralytics' full-val numbers; ranking calibration methods within ~0.6 points is not reliable at this size.
- **Calibration data** comes from val2017 (disjoint from eval, overlap 0), not train2017.
- **PTQ only**, no QAT. **reduce_range's gain is consistent with U8×S8 saturation on a non-VNNI CPU but not proven**; the device check (D4) is 10 images.
- **Kaggle CPU is not an edge device** (Intel(R) Xeon(R) CPU @ 2.00GHz, AVX-512 without VNNI). CPU and NPU latencies are different hardware classes; the comparison that matters is FP32 vs INT8 within each.
- **On-device accuracy:** mAP is from local ORT eval of the same ONNX. On the device, only 10-image detection parity was measured, and device INT8 matches local INT8 at 86%, so device mAP could differ.
- **AI Hub numbers** are for one device (Samsung Galaxy S24 (Family)); peak memory is process-level, not model-level; the FP32 model's internal precision on the NPU is not reported.
- **Box drift** (box MAE) averages over all 8,400 anchors, mostly background, so it overstates the error on boxes that matter.
- Half-pixel scale-back fix measured at +0.0010 mAP50-95 (Q4).

## What I'd do next
- On-device mAP over the 500 images (batched inference jobs), then a QAT run to close the remaining 0.015 mAP.
- Split `output0` into separate box and score outputs at export, so the head can be quantized without the shared-scale collapse.
- Mixed-precision search (only the sensitive layers in FP16), and the same study on yolov8s/m.


In [27]:
# R4 — Interview Q&A (answers cite results.json)
md(f"""## Interview Q&A
1. **Why does full INT8 destroy YOLOv8?** `output0` concatenates boxes (up to {dg['fp32_box_range'][1]:.0f}) and scores (≤ {dg['fp32_score_range'][1]:.2f}). One uint8 scale of {dg['output_scale']:.2f} means every score is within half a step of 0, so all scores become 0 and the model gives {dg['dets_50img_int8_full']} detections on 50 images vs {dg['dets_50img_fp32']} in FP32 ({c_dg}). This holds for MinMax, Entropy and Percentile ({c_col}).
2. **Why wasn't excluding the final Concat enough?** In E1 the box branch still has its own 0–640 tensor at {e1_box:.2f} px per step (scores get {e1_score:.4f}), so localization suffers: mAP50-95 {relA('E1'):+.1f}% vs mAP50 {100*(AB['E1']['map50']-AB['A']['map50'])/AB['A']['map50']:+.1f}% ({c_e1}). Keeping the whole decode tail in FP32 (E2) cut the loss to {relA('E2'):+.1f}%.
3. **QDQ, per-channel, symmetric vs asymmetric: what did you use?** QDQ format; per-channel symmetric int8 weights; asymmetric uint8 activations. Per-channel needs the `axis` attribute on Q/DQ, which needs opset ≥ 13 ({c_op}).
4. **How did calibration methods compare?** With E2: MinMax {AB['E2']['map50_95']:.4f}, Entropy {AB['C_E2']['map50_95']:.4f}, Percentile {AB['D_E2']['map50_95']:.4f}. Within about half a point, too close to rank on 500 images. Percentile had the lowest box drift ({AB['D_E2']['drift_50img']['box_mae_px']:.2f} px).
5. **Did reduce_range matter?** On the CPU, yes: E2 {AB['E2']['map50_95']:.4f} → E2_RR {AB['E2_RR']['map50_95']:.4f}. That's consistent with U8×S8 saturation in `vpmaddubsw` on a CPU without VNNI (VNNI={isa['avx512_vnni']}). On the NPU there's no sign it helps: device-vs-FP32 match E2 {par['int8E2_device_vs_fp32_local']['match_pct']:.0f}% vs E2_RR {par['int8E2RR_device_vs_fp32_local']['match_pct']:.0f}% (10 images).
6. **Why was INT8 barely faster on the CPU?** E2_RR: {cpux('E2_RR','t1'):.2f}× at 1 thread, {cpux('E2_RR',tmax):.2f}× at {tmax[1:]} threads. No VNNI, Q/DQ overhead around the FP32 tail, and yolov8n is small.
7. **And on the NPU?** FP32 {prof['fp32']['inference_ms']:.2f} ms → INT8 {prof['int8_E2']['inference_ms']:.2f} ms ({npux:.2f}×) on {devc['name']} ({c_prof}).
8. **What fell back to CPU on Snapdragon?** Nothing: {prof['int8_E2']['layers_per_unit'].get('NPU',0)}/{prof['int8_E2']['layers_total']} INT8 layers on the NPU, including the float decode tail.
9. **How did you verify the phone matches your desktop pipeline?** I ran 10 eval images on-device and applied my own decode + NMS to the raw outputs: FP32 {par['fp32_device_vs_fp32_local']['match_pct']:.0f}% detections matched, box MAE {par['fp32_device_vs_fp32_local']['raw_box_mae_px']:.2f} px ({c_par}).
10. **Why does INT8 on the device differ from INT8 in ORT?** Different quantized kernels (rounding, requantization, float precision in the tail): {par['int8E2_device_vs_int8E2_local']['match_pct']:.0f}% match. The same INT8 model doesn't mean the same arithmetic.
11. **Why can calibration run out of memory when inference doesn't?** It keeps every activation of every image: {mem['mb_per_image']} MB × {sub['calib_images']} = {mem['gb_for_all_calib']} GB. I chunked it and proved MinMax chunked is bit-identical to single-pass ({c_chk}).
12. **Why must calibration and eval sets be disjoint?** Otherwise ranges are tuned on test data and INT8 accuracy is inflated. Here: {sub['calib_images']} calibration / {sub['eval_images']} eval images, overlap {sub['overlap']} ({c_sub}).
13. **Why is your FP32 mAP above Ultralytics' published number?** It's a 500-image subset, not full val2017: n {fp['n']['map50_95']:.3f}, s {fp['s']['map50_95']:.3f}, m {fp['m']['map50_95']:.3f}. The claim is "in the expected range", not "beats published".
""")

## Interview Q&A
1. **Why does full INT8 destroy YOLOv8?** `output0` concatenates boxes (up to 637) and scores (≤ 0.93). One uint8 scale of 2.54 means every score is within half a step of 0, so all scores become 0 and the model gives 0 detections on 50 images vs 6963 in FP32 (Q10). This holds for MinMax, Entropy and Percentile (Q13).
2. **Why wasn't excluding the final Concat enough?** In E1 the box branch still has its own 0–640 tensor at 2.57 px per step (scores get 0.0038), so localization suffers: mAP50-95 -25.6% vs mAP50 -15.0% (Q13). Keeping the whole decode tail in FP32 (E2) cut the loss to -11.6%.
3. **QDQ, per-channel, symmetric vs asymmetric: what did you use?** QDQ format; per-channel symmetric int8 weights; asymmetric uint8 activations. Per-channel needs the `axis` attribute on Q/DQ, which needs opset ≥ 13 (Q6b).
4. **How did calibration methods compare?** With E2: MinMax 0.3368, Entropy 0.3421, Percentile 0.3428. Within about half a point, too close to rank on 500 images. Percentile had the lowest box drift (4.98 px).
5. **Did reduce_range matter?** On the CPU, yes: E2 0.3368 → E2_RR 0.3663. That's consistent with U8×S8 saturation in `vpmaddubsw` on a CPU without VNNI (VNNI=False). On the NPU there's no sign it helps: device-vs-FP32 match E2 90% vs E2_RR 88% (10 images).
6. **Why was INT8 barely faster on the CPU?** E2_RR: 1.16× at 1 thread, 0.96× at 4 threads. No VNNI, Q/DQ overhead around the FP32 tail, and yolov8n is small.
7. **And on the NPU?** FP32 3.34 ms → INT8 2.19 ms (1.52×) on Samsung Galaxy S24 (Family) (D3).
8. **What fell back to CPU on Snapdragon?** Nothing: 251/251 INT8 layers on the NPU, including the float decode tail.
9. **How did you verify the phone matches your desktop pipeline?** I ran 10 eval images on-device and applied my own decode + NMS to the raw outputs: FP32 100% detections matched, box MAE 0.44 px (D4).
10. **Why does INT8 on the device differ from INT8 in ORT?** Different quantized kernels (rounding, requantization, float precision in the tail): 86% match. The same INT8 model doesn't mean the same arithmetic.
11. **Why can calibration run out of memory when inference doesn't?** It keeps every activation of every image: 235.5 MB × 300 = 69.0 GB. I chunked it and proved MinMax chunked is bit-identical to single-pass (Q9).
12. **Why must calibration and eval sets be disjoint?** Otherwise ranges are tuned on test data and INT8 accuracy is inflated. Here: 300 calibration / 500 eval images, overlap 0 (Q3).
13. **Why is your FP32 mAP above Ultralytics' published number?** It's a 500-image subset, not full val2017: n 0.381, s 0.463, m 0.512. The claim is "in the expected range", not "beats published".


In [28]:
# R5 — Claim ledger: a resume claim exists only if it has a row here
ledger = [
    ("INT8 post-training static quantization (QDQ, per-channel) with a 300-image calibration set, 500-image COCO eval",
     f"best E2_RR mAP50-95 {AB['A']['map50_95']:.4f} → {best['map50_95']:.4f}; size {AB['A']['size_mb']:.2f} → {best['size_mb']:.2f} MB", f"{c_best}, {ABC['E2_RR']}"),
    ("Ablated calibration method, head exclusion and reduce_range",
     f"B {AB['B']['map50_95']:.3f}, E1 {AB['E1']['map50_95']:.3f}, E2 {AB['E2']['map50_95']:.3f}, Entropy {AB['C_E2']['map50_95']:.3f}, Percentile {AB['D_E2']['map50_95']:.3f}, E2_RR {AB['E2_RR']['map50_95']:.3f}", "Q12, Q14"),
    ("Diagnosed full-INT8 collapse of YOLOv8's output",
     f"{AB['B']['n_dets']} dets; shared output scale {dg['output_scale']:.2f} vs max score {dg['fp32_score_range'][1]:.2f}", f"{c_dg}, {c_col}"),
    ("Memory-bounded chunked calibration, validated against single-pass",
     f"{mem['gb_for_all_calib']} GB → chunks of 25; MinMax max rel diff {chk['MinMax']['max_rel_diff']}", f"{c_mem}, {c_chk}"),
    ("Profiled FP32 vs INT8 on a Snapdragon 8 Gen 3 (Galaxy S24) via Qualcomm AI Hub",
     f"{prof['fp32']['inference_ms']:.2f} → {prof['int8_E2']['inference_ms']:.2f} ms; {prof['int8_E2']['layers_per_unit'].get('NPU',0)}/{prof['int8_E2']['layers_total']} layers on NPU, no CPU fallback", c_prof),
    ("Verified on-device outputs against the desktop pipeline",
     f"FP32 {par['fp32_device_vs_fp32_local']['match_pct']:.0f}% detections matched on 10 images", c_par),
    ("Benchmarked YOLOv8 n/s/m (mAP, size, CPU latency)",
     f"mAP {fp['n']['map50_95']:.3f}/{fp['s']['map50_95']:.3f}/{fp['m']['map50_95']:.3f}; t1 {latf['n']['t1']['mean_ms']:.0f}/{latf['s']['t1']['mean_ms']:.0f}/{latf['m']['t1']['mean_ms']:.0f} ms", f"Q4, {c_lat}"),
]
md("## Claim ledger\n| Resume claim | Evidence (from results.json) | Cell |\n|---|---|---|\n" +
   "\n".join(f"| {a} | {b} | {c} |" for a, b, c in ledger))
json.dump([{"claim": a, "evidence": b, "cells": c} for a, b, c in ledger],
          open(f"{WORK}/claim_ledger_nb3.json", "w"), indent=2)
print("claim_ledger_nb3.json written")

## Claim ledger
| Resume claim | Evidence (from results.json) | Cell |
|---|---|---|
| INT8 post-training static quantization (QDQ, per-channel) with a 300-image calibration set, 500-image COCO eval | best E2_RR mAP50-95 0.3812 → 0.3663; size 12.24 → 3.49 MB | Q14, Q14 |
| Ablated calibration method, head exclusion and reduce_range | B 0.000, E1 0.284, E2 0.337, Entropy 0.342, Percentile 0.343, E2_RR 0.366 | Q12, Q14 |
| Diagnosed full-INT8 collapse of YOLOv8's output | 0 dets; shared output scale 2.54 vs max score 0.93 | Q10, Q13 |
| Memory-bounded chunked calibration, validated against single-pass | 69.0 GB → chunks of 25; MinMax max rel diff 0.0 | Q8, Q9 |
| Profiled FP32 vs INT8 on a Snapdragon 8 Gen 3 (Galaxy S24) via Qualcomm AI Hub | 3.34 → 2.19 ms; 251/251 layers on NPU, no CPU fallback | D3 |
| Verified on-device outputs against the desktop pipeline | FP32 100% detections matched on 10 images | D4 |
| Benchmarked YOLOv8 n/s/m (mAP, size, CPU latency) | mAP 0.381/0.463/0.512; t1 99/283/738 ms | Q4, Q5 |

claim_ledger_nb3.json written
